In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
import imageio.v2 as imageio
import tempfile
from IPython.display import Markdown, display, Video
import os
import time

import sys
sys.path.extend(["../../"])

from utils import squared_loss_function, absolute_loss_function, activation_function_linear
from utils import computation_graph_linear, create_computation_graph_linear
from plot_utils import Arrow3D
from utils import fit_norm2_least_square
from configs import cg_rl_RR as cg

In [ ]:
display(Markdown(open("../../_macros.md").read()))

# Linear Regression

This document shows the theory behind linear regression. It is intended to be an incremental document where I will be uploading different information on demand.

Many people like seeing Linear Regression as a special model in itself. I like to avoid this categorization. For me, linear regression is no more than a specific type of neural network that, due to its construction, allows us to derive many theoretical results that are not possible for other models. In fact, linear regression is not really a linear model in the sense of learning to represent data through high-dimensional hyperplanes, but a model that is a linear model over
basis functions. Only when the basis function is the identity do these models learn hyperplanes.

We can talk about four types of linear regression models:

* $f: \mathbb{R} \rightarrow \mathbb{R}$
* $f: \mathbb{R}^D \rightarrow \mathbb{R}$
* $f: \mathbb{R} \rightarrow \mathbb{R}^C$
* $f: \mathbb{R}^D \rightarrow \mathbb{R}^C$


## One dimensional linear regression : $f: \mathbb{R} \rightarrow \mathbb{R}$

One-dimensional linear regression stands for problems where, for an input $x\in \mathbb{R}$, we want to predict an output $t\in \mathbb{R}$, assuming some form of noise model. This, however, will be covered later in an advanced section, named a probabilistic perspective of linear regression.

The idea is to predict some target $t$ assuming that there is a linear relation between the inputs $x$. So mathematically, we want to model:

$$
y = w \cdot x + b
$$

with $w \in \mathbb{R}, b\in \mathbb{R}$. Here $y$ represents the prediction made for $x$, and we want this prediction to be $t$.

The goal in linear regression is to find the linear model that best represents some given data. In other words, it is a search problem where we want to find the values of $w,b$  that better represent some given data. $w,b$ are known as the parameters because they are the elements that parameterize the function.

Let's assume we have three different points $\{(x_n,t_n\}^3_{n=1}$ representing $x$ altura and $t$ peso. So our goal is to find a linear relationship between the altura and peso of some people. Assume we have these values:

$$
\begin{split}
(x_1,t_1) &= (0,0.2)\\
(x_2,t_2) &= (1,0.5)\\
(x_3,t_3) &= (2,2.8)\\
\end{split}
$$

Let's plot these values:

In [ ]:
# input to our model. Represents time in seconds
x_data = np.array([0,1,2]).reshape(3,1)
# outputs associated to each input. Represents cantidad de lluvia in mm^3
t_data = np.array([0.2,0.5,2.8]).reshape(3,1)

## display
plt.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')
for i in range(len(x_data)):
    plt.text(x_data[i][0], t_data[i][0] + 0.1, f"$(x_{i+1},t_{i+1})=({x_data[i][0]},{t_data[i][0]})$", fontsize=8, ha="center")  # Etiqueta sobre el punto
plt.xlabel('altura')
plt.ylabel('peso')
plt.ylim([cg.data_y_lim_l,cg.data_y_lim_u])
plt.xlim([cg.data_x_lim_l,cg.data_x_lim_u])
plt.legend()

### Displaying Linear Models

There are many possible linear models that can explain this data. In particular, there are $\infty$ possible values for $w$ and $b$. Let's plot 3 possible linear models.


In [ ]:
## ======================================================================= ##
## display possible functions depending on different values of $w$ and $b$ ##
## ======================================================================= ##

## fix seed so that randomness is controlled.
np.random.seed(cg.seed)

## number of points in the domain used to plot the functions 
N_points_domain = cg.N_domain_x

## display data again
plt.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')
for i in range(len(x_data)):
    plt.text(x_data[i][0], t_data[i][0] + 0.1, f"$(x_{i+1},t_{i+1})=({x_data[i][0]},{t_data[i][0]})$", fontsize=8, ha="center")  # Etiqueta sobre el punto
plt.xlabel('altura')
plt.ylabel('peso')
plt.ylim([cg.data_y_lim_l,cg.data_y_lim_u])
plt.xlim([cg.data_x_lim_l,cg.data_x_lim_u])


## ===========================================
## Neural network specification for each layer

# neurons of input layer
n_in = 1
# neurons of output layer
n_out = 1

## ================================================================================
## Create several possible functions that our specific neural network can implement
for i in range(3):

    # domain over where we want to plot the function implemented by the NNet
    x_range = np.linspace(cg.data_x_range_l,cg.data_x_range_u, N_points_domain).reshape((N_points_domain,1))

    # initialize one of our networks
    w, b = create_computation_graph_linear(n_in,n_out)

    # projection from input x to output y through computational graph
    y_range = computation_graph_linear(x_range,w,b)

    # check how this computational_graph predicts at the inputs denote by our observed data X.
    y_pred = computation_graph_linear(x_data,w,b)

    if i == 0:
        plt.plot(x_range,y_range, color = f"C{i+1}", label = 'linear function' )
    else:
        plt.plot(x_range,y_range, color = f"C{i+1}")

plt.legend(loc = 'upper left')

### Loss functions

The question now is: what do we understand by optimal parameters $w,b$ representing our data?. Obviously, if the relationship between $x$ and $t$ is totally linear, then the line that goes through all the points is the best possible representation.

However, if the data does not have a perfect linear relationship, as in our example, then there is no line that can go through these points. Here is where the concept of a loss function comes into play.

First of all, note that for different possible models (orange, red, and green lines), we will have different predictions for each of our data points. We label predictions using $y$. So $t$ is our target and $y$ is the prediction of the model.

For the first candidate linear model $a$ with parameters $w_a, b_a$. The predictions for each of our three points are: 

$$
\begin{split}
y_1 &= w_a \cdot x_1 + b_a \\
y_2 &= w_a \cdot x_2 + b_a \\
y_3 &= w_a \cdot x_3 + b_a \\
\end{split}
$$

Similarly for the other two candidates $b$ and $c$ we have:

$$
\begin{split}
y_1 = w_b \cdot x_1 + b_b \\
y_2 = w_b \cdot x_2 + b_b \\
y_3 = w_b \cdot x_3 + b_b \\
\end{split}
$$

$$
\begin{split}
y_1 = w_c \cdot x_1 + b_c\\
y_2 = w_c \cdot x_2 + b_c \\
y_3 = w_c \cdot x_3 + b_c \\
\end{split}
$$

Let's plot these predictions alongside the lines:

In [ ]:
## fix seed so that randomness is controlled.
np.random.seed(cg.seed)

## number of points in the domain used to plot the functions 
N_points_domain = cg.N_domain_x

## display data again
plt.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')
for i in range(len(x_data)):
    plt.text(x_data[i][0], t_data[i][0] - 0.25, f"$(x_{i+1},t_{i+1})=({x_data[i][0]},{t_data[i][0]})$", fontsize=8, ha="center")  
plt.xlabel('altura')
plt.ylabel('peso')
plt.ylim([cg.losses_named['squared'].loss_y_lim_l,cg.losses_named['squared'].loss_y_lim_u])
plt.xlim([cg.losses_named['squared'].loss_x_lim_l,cg.losses_named['squared'].loss_x_lim_u])

## ================================================================================
## Create several possible functions that our specific neural network can implement
for i in range(3):

    # initialize one of our networks
    w, b = create_computation_graph_linear(n_in,n_out)

    # projection from input x to output y through computational graph
    y_range = computation_graph_linear(x_range,w,b)

    # check how this computational_graph predicts at the inputs denote by our observed data X.
    y_pred = computation_graph_linear(x_data,w,b)

    if i == 0:
        plt.plot(x_data, y_pred,'*', markersize = 5, color = f"C{i+1}", label = 'Predictions at training input data')
        plt.plot(x_range,y_range, color = f"C{i+1}", label = 'function on all the domain' )
        
        for j in range(len(x_data)):
            plt.text(x_data[j][0], y_pred[j][0] - 0.25, f"$(x_{j+1},y_{j+1})=({x_data[j][0]:.2f},{y_pred[j][0]:.2f})$", fontsize=8, ha="center", color = f"C{i+1}") 
    else:
        plt.plot(x_data, y_pred,'*', markersize = 5,  color = f"C{i+1}")
        plt.plot(x_range,y_range, color = f"C{i+1}")
        
        for j in range(len(x_data)):
            plt.text(x_data[j][0], y_pred[j][0] + 0.25, f"$(x_{j+1},y_{j+1})=({x_data[j][0]:.2f},{y_pred[j][0]:.2f})$", fontsize=8, ha="center", color = f"C{i+1}") 

plt.legend()



As we see from the plot, the prediction made by the red line at the third point $(x_3,t_3)$ is perfect. However, it does not go near the points $(x_1,t_1)$ and $(x_2,t_2)$. For these two points, the orange line is a better candidate.   

If you look at the picture, which line would you choose to use?. Intuitively, we will be tempted to select the line that is the closest possible to all the points. So we want a line that is the best possible explanation on average for all the points. So in this example, although the red line is a good candidate for the third point, we will be tempted to select the orange line because it better represents the three points.

Next, we should ask ourselves: what do we understand from the "closest" possible line?. We need to transform this into a mathematical expression because machine learning is about math. It turns out that there are many ways to define closeness in mathematical spaces. Actually, the way we define closeness induces different mathematical spaces. For this, we need to introduce the concept of distances. If we are able to measure the distance between $t$ and $y$, we can select the line that has the closest possible distance.

In mathematics a distance $d : \mathbb{R}^N \times \mathbb{R}^N \rightarrow \mathbb{R}$ is a function that satisfies:

$$
\begin{split}
d(x,x) = 0 \\
d(x,y) > 0, \, x\neq y \\
d(y,x) = d(x,y) \\
d(x,z) \leq d(x,y) + d(x,z)
\end{split}
$$


We are going to use two different distances, the square of the Euclidean distance and the Manhattan distance. The reason why we used the square of the Euclidean distance and not the standard distance is given in the advanced section, which covers the probabilistic perspective. However, I will show here that any constant modification to a loss function does not change the result of the optimization process.

* Square Euclidean distance: $ d(a,b) = (a-b)^2$
* Manhatan distance: $ d(a,b) = |a-b|$

As we can see, the Manhattan distance measures the absolute value of the difference between two points.

Thus, we can now measure the distance between the model's predictions $y$ and the desired targets $t$ on each of our training points and for each of our models. Let's look at it:


In [ ]:
## fix seed so that randomness is controlled.
np.random.seed(cg.seed)

## number of points in the domain used to plot the functions 
N_points_domain = cg.N_domain_x

## display data again
fig, (ax1,ax2) = plt.subplots(1,2, figsize=(10,5))

ax1.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')
ax2.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

ax1.set_xlabel('altura')
ax1.set_ylabel('peso')
ax1.set_title('Squared Euclidean Distance')
ax1.set_ylim([cg.losses_named['squared'].loss_y_lim_l,cg.losses_named['squared'].loss_y_lim_u])
ax1.set_xlim([cg.losses_named['squared'].loss_x_lim_l,cg.losses_named['squared'].loss_x_lim_u])

ax2.set_xlabel('altura')
ax2.set_ylabel('peso')
ax2.set_title('Manhatan Distance')
ax2.set_ylim([cg.losses_named['absolute'].loss_y_lim_l,cg.losses_named['absolute'].loss_y_lim_u])
ax2.set_xlim([cg.losses_named['absolute'].loss_x_lim_l,cg.losses_named['absolute'].loss_x_lim_u])

## ================================================================================
## Create several possible functions that our specific neural network can implement
for i, model_id in zip(range(3),['a','b','c']):

    # initialize one of our networks
    w, b = create_computation_graph_linear(n_in,n_out)

    # projection from input x to output y through computational graph
    y_range = computation_graph_linear(x_range,w,b)

    # check how this computational_graph predicts at the inputs denote by our observed data X.
    y_pred = computation_graph_linear(x_data,w,b)

    squared_loss = squared_loss_function(t_data, y_pred)
    absolute_loss = absolute_loss_function(t_data, y_pred)
    

    if i == 0:
        ax1.plot(x_data, y_pred,'*', markersize = 5, color = f"C{i+1}", label = 'Predictions at training input data')
        ax1.plot(x_range,y_range, color = f"C{i+1}", label = 'function on all the domain' )
        
        for j in range(len(x_data)):
            ax1.text(x_data[j], y_pred[j] - 0.25, f"$d(y_{j+1},t_{j+1})={float(squared_loss[j][0]):.2}$", fontsize=8, ha="center", color = f"C{i+1}") 
            
        ax1.text(1,-0.25*i, f"$L(w_{model_id},b_{model_id}) = {float(np.sum(squared_loss)):.2f}$", color = f"C{i+1}")    
            
        ax2.plot(x_data, y_pred,'*', markersize = 5, color = f"C{i+1}", label = 'Predictions at training input data')
        ax2.plot(x_range,y_range, color = f"C{i+1}", label = 'function on all the domain' )
        
        for j in range(len(x_data)):
            ax2.text(x_data[j], y_pred[j] - 0.25, f"$d(y_{j+1},t_{j+1})={float(absolute_loss[j][0]):.2}$", fontsize=8, ha="center", color = f"C{i+1}") 
            
        ax2.text(1,-0.25*i, f"$L(w_{model_id},b_{model_id}) = {float(np.sum(absolute_loss)):.2f}$", color = f"C{i+1}")    
        
    else:
        ax1.plot(x_data, y_pred,'*', markersize = 5,  color = f"C{i+1}")
        ax1.plot(x_range,y_range, color = f"C{i+1}")
        
        for j in range(len(x_data)):
            ax1.text(x_data[j], y_pred[j] + 0.25, f"$d(y_{j+1},t_{j+1})={float(squared_loss[j][0]):.2}$", fontsize=8, ha="center", color = f"C{i+1}")
            
        ax1.text(1,-0.25*i, f"$L(w_{model_id},b_{model_id}) = {float(np.sum(squared_loss)):.2f}$", color = f"C{i+1}")    
        

        ax2.plot(x_data, y_pred,'*', markersize = 5,  color = f"C{i+1}")
        ax2.plot(x_range,y_range, color = f"C{i+1}")
        
        for j in range(len(x_data)):
            ax2.text(x_data[j], y_pred[j] + 0.25, f"$d(y_{j+1},t_{j+1})={float(absolute_loss[j][0]):.2}$", fontsize=8, ha="center", color = f"C{i+1}")
            
 
        ax2.text(1,-0.25*i, f"$L(w_{model_id},b_{model_id}) = {float(np.sum(absolute_loss)):.2f}$", color = f"C{i+1}") 

We can now observe what we have intuitively reasoned. The first thing is that the red line, on the third point, has the lowest distance, being closest to $0$ in the case of the squared Euclidean distance. However, the loss on the rest of the points is higher than when compared to the orange line.

Actually, the notion of *orange line* being a better representation of the points can be seen mathematically by the expected distances. In other words, by summing the distances of each point and dividing by the number of points. These distances, in machine learning, are called losses, and the average of individual losses is called the expected loss. In other words:

$$
\begin{split}
L(w_a,b_a) = d(y_1(w_a,b_a), t_1) + d(y_2(w_a,b_a), t_2) + d(y_3(w_a,b_a), t_3) \\
L(w_b,b_b) = d(y_1(w_b,b_b), t_1) + d(y_2(w_b,b_b), t_2) + d(y_3(w_b,b_b), t_3) \\
L(w_c,b_c) = d(y_1(w_c,b_c), t_1) + d(y_2(w_c,b_c), t_2) + d(y_3(w_c,b_c), t_3) \\
\end{split}
$$

where, since $y$ is a function of the parameters, I am using the notation $y(w_a,b_a,x)$ to denote the linear function, since $y_1(w_a,b_a,x) = w_a\cdot x_1 + b_a $. 

From now on, we will consider just the sum without dividing by the number of points. As we shall see later, in our context, it is equivalent. Whether we take the sum, the average, or divide by another number depends on what we are actually trying to represent, and in other contexts, such as Bayesian inference, it makes a difference. This depends on how we view machine learning. For example, if we see it through the lens of Bayes' decision rule, then we might be interested in optimizing the expected risk, and this expected risk is the average of the individual sums. Bayesian inference or maximum log-likelihood deals with a sum with possible additional normalization terms, such as when the likelihood function is Gaussian.

In our example, we have (for the Squared Euclidean loss):

$$
\begin{split}
L(w_a,b_a) = 0.22 + 0.37 + 1.6 = 2.15\\
L(w_b,b_b) = 5.7 + 4.8 + 2.9e^{-6} = 10.48\\
L(w_c,b_c) = 0.3 + 7.2 + 7.9 = 15.37\\
\end{split}
$$

which are also displayed in the figure, in addition to the absolute loss, obtained from the Manhattan distance. We see that the overall loss function is lower for the orange line than the red line, and the worst possible representation is the green line. Even though the absolute and squared losses are different for each of the models, both losses provide the same order of which is the best and worst model.

Importantly, the loss functions are functions of the data: $x,t$ and the parameters of the model $w,b$.

We can generalize the concepts seen so far for any arbitrary number of data points $N$. Now we have a dataset $(x_n,t_n)^N_{n=1}$. For a given parameter $w$, the two loss functions we have seen so far are given by:

$$
\begin{split}
L_\text{squared}(w,b) = \sum_{n=1}^N (t_n - y_n)^2 \\
L_\text{absolute}(w,b) = \sum_{n=1}^N \left|t_n - y_n\right| \\
\end{split}
$$

where $y_n = w \cdot x_n + b$. As we mentioned earlier, different parameters will imply a different loss function value. So loss functions can be seen as functions of the parameters. In our example above, we showed the loss function corresponding to three lines. However, we can analyze the loss function value for any parameter and plot it. 

To make the reader familiar, let's look at the loss function varying the weight parameter, fixing the bias parameter to the value of $0.5$. In other words, we show:

$$
\begin{split}
L_\text{squared}(w,b = 0.5,x,t) = \sum_{n=1}^N (t_n - y_n)^2 \\
L_\text{absolute}(w,b = 0.5,x,t) = \sum_{n=1}^N \left|t_n - y_n\right| \\
\end{split}
$$

where now: $y_n = w \cdot x_n + 0.5$.

In [ ]:
## ============================================================================== ##
## display loss as a function of weight parameter (loss incurred by each network) ##
## ============================================================================== ##
## Let's see the associated loss to each possible function but seeing the loss
## as a function of the weight parameter. To do so we fix the bias to 0.
## We show two different losses: squared (top) and absolute ( bottom )
## I repeat code from above but computing and plotting the loss.

## fix seed so that randomness is controlled.
np.random.seed(cg.seed)

## number of points in the domain used to plot the functions 
N_points_domain = cg.N_domain_x

## create figure box
fig, ((ax11,ax12),(ax21,ax22)) = plt.subplots(2,2, figsize = (10,10))


## ===========================================
## Neural network specification for each layer

# neurons of input layer
n_in = 1
# neurons of output layer
n_out = 1
# bias fixed value to plot only depending on w
fix_bias = 0

## ================================================================================
## Create several possible functions that our specific neural network can implement

# to save individual losses, expected losses and parameters used
squared_loss_acc = []
absolute_loss_acc = []
expected_squared_loss_acc = []
expected_absolute_loss_acc = []
w_acc = []
w_range = []
# Compute the loss function over 100 possible models.
for i in range(cg.N_models_simulation):
    
    # domain over where we want to plot the function implemented by the NNet
    x_range = np.linspace(cg.data_x_range_l,cg.data_x_range_u, N_points_domain).reshape((N_points_domain,1))

    # initialize one of our networks
    w, b = create_computation_graph_linear(n_in,n_out)

    # projection from input x to output y through computational graph
    y_range = computation_graph_linear(x_range,w, b = fix_bias)

    # check how this computational_graph predicts at the inputs denote by our observed data X.
    y_pred = computation_graph_linear(x_data,w, b = fix_bias)

    # compute the two losses at the predictions
    squared_loss = squared_loss_function(t_data, y_pred)
    absolute_loss = absolute_loss_function(t_data, y_pred)

    # acumulate loss and parameter used
    squared_loss_acc.append(squared_loss)
    absolute_loss_acc.append(absolute_loss)
    expected_squared_loss_acc.append(np.sum(squared_loss))
    expected_absolute_loss_acc.append(np.sum(absolute_loss))
    w_range.append(np.squeeze(w))
    w_acc.append(w)

    
# sort loss and weights to interactive plot later
idx = np.argsort(w_range)
sorted_w_range = np.array(w_range)[idx]
sorted_expected_squared_loss_acc = np.array(expected_squared_loss_acc)[idx]
sorted_expected_absolute_loss_acc = np.array(expected_absolute_loss_acc)[idx]

## Display different models sequentially, alongside its loss.

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")


# variables to keep track of old weight losses display to show the overall loss function.
w_old = []
expected_squared_loss_old = []
expected_absolute_loss_old = []
for i,w in zip(range(cg.N_models_simulation),w_acc):  
    
    ## repeat the projection from input x to output y through computational graph
    y_range = computation_graph_linear(x_range,w, b = fix_bias)

    # check how this computational_graph predicts at the inputs denote by our observed data X.
    y_pred = computation_graph_linear(x_data,w, b = fix_bias)
    
    ## for subsequent plotting
    w = np.squeeze(w)
    
    ## clean up points     
    ax11.cla()
    ax12.cla()
    ax21.cla()
    ax22.cla()

    ## display data 
    ax11.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')
    ax21.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

    ax11.set_xlabel('altura')
    ax11.set_ylabel('peso')
    ax11.set_title('Squared Euclidean Distance')
    ax11.set_ylim([cg.losses_named['squared'].loss_y_lim_l,cg.losses_named['squared'].loss_y_lim_u])
    ax11.set_xlim([cg.losses_named['squared'].loss_x_lim_l,cg.losses_named['squared'].loss_x_lim_u])

    ax21.set_xlabel('altura')
    ax21.set_ylabel('peso')
    ax21.set_title('Manhatan Distance')
    ax21.set_ylim([cg.losses_named['absolute'].loss_y_lim_l,cg.losses_named['absolute'].loss_y_lim_u])
    ax21.set_xlim([cg.losses_named['absolute'].loss_x_lim_l,cg.losses_named['absolute'].loss_x_lim_u])
    
    ## display loss functions
    ax12.plot(sorted_w_range, sorted_expected_squared_loss_acc )
    ax12.set_xlabel('weight values')
    ax12.set_ylabel('squared loss function')

    ax22.plot(sorted_w_range, sorted_expected_absolute_loss_acc )
    ax22.set_xlabel('weight values')
    ax22.set_ylabel('absolute loss function')

    ## Plot function
    ax11.plot(x_data, y_pred,'*', markersize = 5, color = f"C1", label = 'Predictions at training input data')
    ax11.plot(x_range,y_range, color = f"C1", label = 'function on all the domain' )

    ax21.plot(x_data, y_pred,'*', markersize = 5, color = f"C1", label = 'Predictions at training input data')
    ax21.plot(x_range,y_range, color = f"C1", label = 'function on all the domain' )

    ## Plot individual losses
    for j in range(len(x_data)):
        ax11.text(x_data[j], -0.5, f"$d(y_{j+1},t_{j+1})={squared_loss_acc[i][j].item():.2}$", fontsize=8, ha="center", color = f"C1") 
    
    for j in range(len(x_data)):
        ax21.text(x_data[j], -0.5, f"$d(y_{j+1},t_{j+1})={absolute_loss_acc[i][j].item():.2}$", fontsize=8, ha="center", color = f"C1") 

    ## Plot expected loss
    ax11.text(0,5, f"$L(w = {w:.2f},b = {fix_bias}) = {expected_squared_loss_acc[i]:.2f}$", color = f"C1")  
    ax11.text(0,4.5, f"$y = {w:.2f} \cdot x + {fix_bias}$", color = f"k")  
    
    ax21.text(0,5, f"$L(w = {w:.2f},b = {fix_bias}) = {expected_absolute_loss_acc[i]:.2f}$", color = f"C1")    
    ax21.text(0,4.5, f"$y = {w:.2f} \cdot x + {fix_bias}$", color = f"k")  

    ## Plot already displayed losses
    ax12.plot(w_old, expected_squared_loss_old, '*', color = 'C0')
    ax22.plot(w_old, expected_absolute_loss_old, '*', color = 'C0')
   
    ## Plot the loss in the loss function view
    ax12.plot(w, expected_squared_loss_acc[i], '*', color = f"C1")
    ax22.plot(w, expected_absolute_loss_acc[i], '*',  color = f"C1")

    ax12.text(w,expected_squared_loss_acc[i]+2.5, f"$L(w = {w:.2f},b = {fix_bias}) = {expected_squared_loss_acc[i]:.2f}$", color = f"C1")    
    ax22.text(w,expected_absolute_loss_acc[i]+0.5, f"$L(w = {w:.2f},b = {fix_bias}) = {expected_absolute_loss_acc[i]:.2f}$", color = f"C1")    

    # save old to display in next figure
    w_old.append(w)
    expected_squared_loss_old.append(expected_squared_loss_acc[i])
    expected_absolute_loss_old.append(expected_absolute_loss_acc[i])
      
    ## Cortesía de chatGPT (desde linea siguiente hasta el final de esta celda):
    ## save images for later display
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

writer.close() 
plt.close()


In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

Now that we have familiarized ourselves with the concept of a loss function and what it looks like, we need to note that for this problem we really have two parameters: $w$ and $b$, and so the loss function is a function that must be displayed in a 3-dimensional plot. 

In other words, we will plot the loss function now 

$$
\begin{split}
L_\text{squared}(w,b,x,t) = \sum_{n=1}^N (t_n - y_n)^2 \\
L_\text{absolute}(w,b,x,t) = \sum_{n=1}^N \left|t_n - y_n\right| \\
\end{split}
$$

with: $y = w\cdot x + b$, varying both bias and weight parameter.


In [ ]:
## ======================================================================================= ##
## display loss as a function of weight and bias parameter (loss incurred by each network) ##
## ======================================================================================= ##
## Let's see the associated loss to each possible function but seeing the loss
## as a function of the weight and bias parameter. 
## We show two different losses: squared (top) and absolute ( bottom )

## fix seed so that randomness is controlled.
np.random.seed(5)

## number of points in the domain used to plot the functions 
N_points_domain = 100

## create figure box
fig = plt.figure(figsize = (10,10))
ax11 = fig.add_subplot(221)
ax12 = fig.add_subplot(222, projection='3d')
ax21 = fig.add_subplot(223)
ax22 = fig.add_subplot(224, projection='3d')

## ===========================================
## Neural network specification for each layer

# neurons of input layer
n_in = 1
# neurons of output layer
n_out = 1

## ================================================================================
## Create several possible functions that our specific neural network can implement

## To do so we need a mesh
w_mesh_squared, b_mesh_squared = np.meshgrid(
    np.linspace(cg.losses_named['squared'].w_range_l_2d,cg.losses_named['squared'].w_range_u_2d,100),
    np.linspace(cg.losses_named['squared'].b_range_l_2d,cg.losses_named['squared'].b_range_u_2d,100)
)
w_mesh_absolute, b_mesh_absolute = np.meshgrid(
    np.linspace(cg.losses_named['absolute'].w_range_l_2d,cg.losses_named['absolute'].w_range_u_2d,100),
    np.linspace(cg.losses_named['absolute'].b_range_l_2d,cg.losses_named['absolute'].b_range_u_2d,100)
)

# reshape x_data and t_data for computations
x_data_expanded = x_data[:,np.newaxis]
t_data_expanded = t_data[:,np.newaxis]

# compute linear projection at all pairs of points, one grid per loss type
y_pred_expanded_squared = w_mesh_squared*x_data_expanded + b_mesh_squared
y_pred_expanded_absolute = w_mesh_absolute*x_data_expanded + b_mesh_absolute

# compute loss
expected_squared_loss_mesh = np.sum(squared_loss_function(t_data_expanded, y_pred_expanded_squared), axis = 0)
expected_absolute_loss_mesh = np.sum(absolute_loss_function(t_data_expanded, y_pred_expanded_absolute), axis = 0)

# to save individual losses, expected losses and parameters used
squared_loss_acc = []
absolute_loss_acc = []
expected_squared_loss_acc = []
expected_absolute_loss_acc = []
w_acc = []
b_acc = []

# Compute the loss funciton over 100 possible neural net.
for i in range(100):
   
    # domain over where we want to plot the function implemented by the NNet
    x_range = np.linspace(-1,4, N_points_domain).reshape((N_points_domain,1))

    # initialize one of our networks
    w, b = create_computation_graph_linear(n_in,n_out)

    # projection from input x to output y through computational graph
    y_range = computation_graph_linear(x_range,w,b)

    # check how this computational_graph predicts at the inputs denote by our observed data X.
    y_pred = computation_graph_linear(x_data,w,b)

    # compute the two losses at the predictions
    squared_loss = squared_loss_function(t_data, y_pred)
    absolute_loss = absolute_loss_function(t_data, y_pred)

    # acumulate loss and parameter used
    squared_loss_acc.append(squared_loss)
    absolute_loss_acc.append(absolute_loss)
    expected_squared_loss_acc.append(np.sum(squared_loss))
    expected_absolute_loss_acc.append(np.sum(absolute_loss))
    w_acc.append(w)
    b_acc.append(b)



## Display different models sequentially, alongside its loss.

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")


# variables to keep track of old weight losses display to show the overall loss function.
w_old = []
b_old = []
expected_squared_loss_old = []
expected_absolute_loss_old = []
for i,w,b in zip(range(100),w_acc,b_acc):  
    
    ## repeat the projection from input x to output y through computational graph
    y_range = computation_graph_linear(x_range,w,b)

    # check how this computational_graph predicts at the inputs denote by our observed data X.
    y_pred = computation_graph_linear(x_data,w,b)
    
    ## for subsequent plotting
    w = np.squeeze(w)
    b = np.squeeze(b)
    
    ## clean up points     
    ax11.cla()
    ax12.cla()
    ax21.cla()
    ax22.cla()

    ## display data 
    ax11.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')
    ax21.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

    ax11.set_xlabel('altura')
    ax11.set_ylabel('peso')
    ax11.set_title('Squared Euclidean Distance')
    ax11.set_ylim([-1,6.3])
    ax11.set_xlim([-0.5,2.5])

    ax21.set_xlabel('altura')
    ax21.set_ylabel('peso')
    ax21.set_title('Manhatan Distance')
    ax21.set_ylim([-1,6.3])
    ax21.set_xlim([-0.5,2.5])
    
    ## display loss functions
    ax12.plot_surface(w_mesh_squared, b_mesh_squared, expected_squared_loss_mesh, cmap = 'gray', alpha = 0.75 )
    ax12.set_xlabel('weight values')
    ax12.set_ylabel('bias values')
    ax12.set_zlabel('squared loss function')

    ax22.plot_surface(w_mesh_absolute, b_mesh_absolute, expected_absolute_loss_mesh, cmap = 'gray', alpha = 0.75)
    ax22.set_xlabel('weight values')
    ax22.set_ylabel('bias values')
    ax22.set_zlabel('absolute loss function')

    ## Plot function
    ax11.plot(x_data, y_pred,'*', markersize = 5, color = f"C1", label = 'Predictions at training input data')
    ax11.plot(x_range,y_range, color = f"C1", label = 'function on all the domain' )

    ax21.plot(x_data, y_pred,'*', markersize = 5, color = f"C1", label = 'Predictions at training input data')
    ax21.plot(x_range,y_range, color = f"C1", label = 'function on all the domain' )

    ## Plot individual losses
    for j in range(len(x_data)):
        ax11.text(x_data[j], -0.5, f"$d(y_{j+1},t_{j+1})={squared_loss_acc[i][j].item():.2}$", fontsize=8, ha="center", color = f"C1") 
    
    for j in range(len(x_data)):
        ax21.text(x_data[j], -0.5, f"$d(y_{j+1},t_{j+1})={absolute_loss_acc[i][j].item():.2}$", fontsize=8, ha="center", color = f"C1") 

    ## Plot expected loss
    ax11.text(0,5, f"$L(w = {w:.2f},b = {b:.2f})= {expected_squared_loss_acc[i]:.2f}$", color = f"C1")  
    ax11.text(0,4.5, f"$y = {w:.2f} \cdot x + {b:.2f}$", color = f"k")  
    
    ax21.text(0,5, f"$L(w = {w:.2f},b = {b:.2f}) = {expected_absolute_loss_acc[i]:.2f}$", color = f"C1")    
    ax21.text(0,4.5, f"$y = {w:.2f} \cdot x + {b:.2f}$", color = f"k")  

    ## Plot already displayed losses
    ax12.plot(w_old, b_old, expected_squared_loss_old, '*', color = 'C0')
    ax22.plot(w_old, b_old, expected_absolute_loss_old, '*', color = 'C0')
   
    ## Plot the loss in the loss function view
    ax12.scatter(w, b, expected_squared_loss_acc[i], color = f"C1")
    ax22.scatter(w, b, expected_absolute_loss_acc[i],  color = f"C1")
    
    ax12.text(w,b,expected_squared_loss_acc[i]+2.5, f"$L(w = {w:.2f},b = {b:.2f}) = {expected_squared_loss_acc[i]:.2f}$", color = f"C1", zorder = 2, ha = 'center')    
    ax22.text(w,b,expected_absolute_loss_acc[i]+0.5, f"$L(w = {w:.2f},b = {b:.2f}) = {expected_absolute_loss_acc[i]:.2f}$", color = f"C1", zorder = 2, ha = 'center')    

    # save old to display in next figure
    w_old.append(w)
    b_old.append(b)
    expected_squared_loss_old.append(expected_squared_loss_acc[i])
    expected_absolute_loss_old.append(expected_absolute_loss_acc[i])
      
    ## Cortesía de chatGPT (desde linea siguiente hasta el final de esta celda):
    ## save images for later display
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

writer.close() 
plt.close()


In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

## Squared Loss

### Optimization: ordinary least square method

Now that we have seen that the expected loss can be seen as a function that represents how well each of the possible models represents the data, the final question is: how do we actually obtain, in a principled way, this best possible model?

The best possible model is the one that has the lowest expected loss, because this implies that it is the model closest to all the data in expectation. So, in other words, we need to find the minimum of the loss function, since that minimum is the set of parameters that has the lowest expected loss, i.e., on average, is the one that best represents the data. 

We know from earlier in school that finding minima in functions means computing the derivative and making the derivative equal to $0$, and finally finding the points that give the minimum.

Let's do that. For things I will add at some point here, I am going to just do it for the squared Euclidean loss function.

We want to find the minimum of a vector-argument scalar-valued function; in other words, a function $L: \mathbb{R}^N \rightarrow \mathbb{R}$, where in this problem $N=2$, since we have two parameters.

To find the minimum, we need to obtain the gradient vector and set it equal to zero. I am going to use a naive, straightforward approach for this process. For an advanced and more general viewpoint of the function being optimized here, check the automatic differentiation reports.

We need to solve the following system of equations:

$$
\begin{split}
\grad{w,b}{} L(w,b) = \begin{pmatrix}\frac{\partial L(w,b)}{\partial w} \\ \frac{\partial L(w,b)}{\partial b} \end{pmatrix} = \begin{pmatrix}0 \\ 0 \end{pmatrix}
\end{split}
$$

Let's compute the two partial derivatives of:

$$
\begin{split}
y_n = w\cdot x_n + b \\
L_\text{squared}(w,b ) = \sum_{n=1}^N (t_n - y_n)^2 
\end{split}
$$

This gives:

$$
\begin{split}
\frac{\partial L(w,b)}{\partial w} = -\sum_{n=1}^N 2\cdot(t_n - w\cdot x_n - b)x_n \\
\frac{\partial L(w,b)}{\partial b} = -\sum_{n=1}^N 2\cdot(t_n - w\cdot x_n - b) 
\end{split}
$$

Now set each of the equations to $0$, and solve the system. Let's do it: 

$$
\begin{split}
\frac{\partial L(w,b)}{\partial b} = 0 \\
\sum_{n=1}^N -2\cdot(t_n - w\cdot x_n - b)  = 0\\
-\cancel{2} \sum_{n=1}^N t_n + \cancel{2}w \sum_{n=1}^N x_n + \cancel{2}\sum_{n=1}^N b = 0 \\
w = \frac{ \sum_{n=1}^N t_n - bN}{\sum_{n=1}^N x_n }
\end{split}
$$

Then:

$$
\begin{split}
\frac{\partial L(w,b)}{\partial w} = 0\\
%
%
\sum_{n=1}^N -2\cdot(t_n - w\cdot x_n - b)x_n  = 0 \\
%
%
%
-\cancel{2}\sum_{n=1}^N t_n x_n + \cancel{2}\cancelto{\frac{ \sum_{n=1}^N t_n - bN}{\sum_{n=1}^N x_n }}{w}\sum_{n=1}^N x_n^2 + \cancel{2} b \sum_{n=1}^N x_n = 0\\ 
%
%
-\sum_{n=1}^N t_n x_n + \frac{ \sum_{n=1}^N t_n - bN}{\sum_{n=1}^N x_n }\sum_{n=1}^N x_n^2 +  b \sum_{n=1}^N x_n = 0\\ 
%
%
-\sum_{n=1}^N t_n x_n\sum_{n=1}^N x_n  + \sum_{n=1}^N t_n\sum_{n=1}^N x_n^2  - bN \sum_{n=1}^N x_n^2 +  b \sum_{n=1}^N x_n \sum_{n=1}^N x_n  = 0\\ 
%
%
%
%
-bN \sum_{n=1}^N x_n^2 +  b \sum_{n=1}^N x_n \sum_{n=1}^N x_n  =  \sum_{n=1}^N t_n x_n\sum_{n=1}^N x_n  - \sum_{n=1}^N t_n\sum_{n=1}^N x_n^2\\
%
%
%
b = \frac{\sum_{n=1}^N t_n x_n\sum_{n=1}^N x_n -\sum_{n=1}^N t_n\sum_{n=1}^N x_n^2}{ \sum_{n=1}^N x_n \sum_{n=1}^N x_n - N \sum_{n=1}^N x_n^2}
%
\end{split}
$$

Finally, we substitute $b$ into the expression of $w$.

$$
w = \frac{ \sum_{n=1}^N t_n - \frac{\sum_{n=1}^N t_n x_n\sum_{n=1}^N x_n -\sum_{n=1}^N t_n\sum_{n=1}^N x_n^2}{ \sum_{n=1}^N x_n \sum_{n=1}^N x_n - N \sum_{n=1}^N x_n^2} N}{\sum_{n=1}^N x_n }
$$


These equations look, obviously, like a nightmare. There are many reasons to try and simplify these expressions. For example:

* Simplifying expressions can reduce the number of computations.
* Simplifying expressions can provide light into statistical relationships between the data and the optimal parameters, which can provide light into further theoretical analysis on things like overfitting, bounds on generalization, etc.

Let's try to do this. First of all, we see that many of the subexpressions involved in the solutions to the optimal parameters imply summations over $x_n$ and $t_n$, or squared versions of $x_n^2$. These summations are Monte Carlo estimates of different well-known expectations:

$$
\begin{split}
\mathbb{E}[x] = \mu_{x} \approx \frac{1}{N} \sum_{n=1}^N x_n \\
\mathbb{E}[x^2]  \approx \frac{1}{N} \sum_{n=1}^N x_n^2 \\ 
\mathbb{V}[x] = \sigma^2_{x} \approx \frac{1}{N} \sum_{n=1}^N x_n^2 - \left( \frac{1}{N} \sum_{n=1}^N x_n \right)^2 = \mathbb{E}[x^2] - \mu_x^2\\
\mathbb{E}[x,y] \approx \frac{1}{N} \sum_{n=1}^N x_n \cdot y_n \\
\mathbb{COV}[x,y] = \sigma_{x,y} \approx \frac{1}{N} \sum_{n=1}^N x_n \cdot y_n - \mu_x \mu_y =  \mathbb{E}[x,y] - \mu_x\mu_y
\end{split}
$$

Let's work out some manipulations on the expressions we have for $w$ and $b$.

$$
\begin{split}
w &= \frac{ \sum_{n=1}^N t_n - \frac{\sum_{n=1}^N t_n x_n\sum_{n=1}^N x_n -\sum_{n=1}^N t_n\sum_{n=1}^N x_n^2}{ \sum_{n=1}^N x_n \sum_{n=1}^N x_n - N \sum_{n=1}^N x_n^2} N}{\sum_{n=1}^N x_n } \frac{\frac{1}{N}}{\frac{1}{N}}\\
%
%
&= \frac{ \mu_t - \frac{\sum_{n=1}^N t_n x_n\sum_{n=1}^N x_n -\sum_{n=1}^N t_n\sum_{n=1}^N x_n^2}{ \sum_{n=1}^N x_n \sum_{n=1}^N x_n - N \sum_{n=1}^N x_n^2}\frac{\frac{1}{N^2}}{\frac{1}{N^2}}}{\mu_x } \\
%
%
%
&= \frac{ \mu_t - \frac{\mathbb{E}[x,t]\mu_x -\mu_t\mathbb{E}[x^2]}{ \mu_x^2 -\mathbb{E}[x^2]}}{\mu_x } \\
%
%
&= \frac{ \mu_t + \frac{\mathbb{E}[x,t]\mu_x -\mu_t\mathbb{E}[x^2]}{ \cancelto{\mathbb{VAR}[X]}{\mathbb{E}[x^2] -\mu_x^2 }}}{\mu_x } \\
%
%
&=  \frac{\mathbb{VAR}[x]\mu_t + \mathbb{E}[x,t]\mu_x -\mu_t\mathbb{E}[x^2]}{ \mathbb{VAR}[x] \mu_x } \\
%
%
&=  \frac{ \mu_t \left( \cancelto{\cancel{\mathbb{E}[x^2]} - \mu_x^2}{\mathbb{VAR}[x]} \cancel{-\mathbb{E}[x^2]} \right) + \mathbb{E}[x,t]\mu_x }{ \mathbb{VAR}[x] \mu_x } \\
%
%
&=  \frac{ -\mu_t\mu_x^2 + \mathbb{E}[x,t]\mu_x }{ \mathbb{VAR}[x] \mu_x } \\
%
%
&=  \frac{ -\mu_t\mu_x + \mathbb{E}[x,t]}{ \mathbb{VAR}[x] } \\
%
%
&=  \frac{ \mathbb{COV}[x,t]}{ \mathbb{VAR}[x] } 
\end{split}
$$

With the bias, we need to follow a similar procedure:

$$
\begin{split}
b &= \frac{\sum_{n=1}^N t_n x_n\sum_{n=1}^N x_n -\sum_{n=1}^N t_n\sum_{n=1}^N x_n^2}{ \sum_{n=1}^N x_n \sum_{n=1}^N x_n - N \sum_{n=1}^N x_n^2}\\
%
%
&= \frac{\sum_{n=1}^N t_n x_n\sum_{n=1}^N x_n -\sum_{n=1}^N t_n\sum_{n=1}^N x_n^2}{ \sum_{n=1}^N x_n \sum_{n=1}^N x_n - N \sum_{n=1}^N x_n^2} \frac{\frac{1}{N^2}}{\frac{1}{N^2}}\\
%
%
&=\frac{\mathbb{E}[x,t]\mu_x - \mu_t \mathbb{E}[x^2]}{\mu_x\mu_x - \mathbb{E}[x^2]} \frac{-1}{-1}\\
%
%
&=\frac{-\cancelto{\mathbb{COV}[x,t]-\mu_x\mu_t}{\mathbb{E}[x,t]}\mu_x + \mu_t \cancelto{\mathbb{VAR}[x]-\mu_x^2}{\mathbb{E}[x^2]}}{\mathbb{VAR}[x]}\\
%
%
&=\frac{-\mu_x\mathbb{COV}[x,t]+\cancel{\mu_x\mu_x\mu_t} + \mu_t \mathbb{VAR}[x]\cancel{-\mu_t\mu_x^2}}{\mathbb{VAR}[x]}\\
%
%
&=\frac{-\mu_x\cancelto{w}{\mathbb{COV}[x,t]}{\mathbb{VAR}[x]}} + \frac{\mu_t \mathbb{VAR}[x]}{\mathbb{VAR}[x]}\\
%
%
&= \mu_t -w\mu_x
\end{split}
$$

So, in summary, we have that the optimal parameters for regression using a squared loss are:

$$
\begin{split}
w = \frac{ \mathbb{COV}[x,t]}{ \mathbb{VAR}[x] }  \\
b = \mu_t -w\mu_x
\end{split}
$$

Let's code this up and see if we get the desired result.

In [ ]:
## optimal parameters
N = x_data.shape[0]

mu_x = 1/N * np.sum(x_data)
mu_t = 1/N * np.sum(t_data)

cov_xt = 1/N * np.sum( x_data * t_data )  - mu_x*mu_t
var_x = 1/N *np.sum(x_data**2) - mu_x**2

w_opt = np.reshape( cov_xt / var_x , (1,1))
b_opt = mu_t - w_opt*mu_x

## get_predictions and function
y_range_opt = computation_graph_linear(x_range,w_opt,b_opt)
y_pred_opt = computation_graph_linear(x_data,w_opt,b_opt)

## get loss
squared_loss_opt = squared_loss_function(t_data, y_pred_opt)

## get loss funciton on all domain
w_mesh, b_mesh = np.meshgrid(np.linspace(cg.losses_named['squared'].w_range_l_2d,cg.losses_named['squared'].w_range_u_2d,100),np.linspace(cg.losses_named['squared'].b_range_l_2d,cg.losses_named['squared'].b_range_u_2d,100))

# reshape x_data and t_data for computations
x_data_expanded = x_data[:,np.newaxis]
t_data_expanded = t_data[:,np.newaxis]

# compute linear projection at all pairs of points
y_pred_expanded = w_mesh*x_data_expanded + b_mesh

# compute loss
expected_squared_loss_mesh = np.sum(squared_loss_function(t_data_expanded, y_pred_expanded), axis = 0)


## Compute optimal loss fixing  bias

# to save individual losses, expected losses and parameters used
expected_squared_loss_acc = []

w_range = []
# Compute the loss funciton over 100 possible neural net.
for i in range(cg.N_models_simulation):
    
    # domain over where we want to plot the function implemented by the NNet
    x_range = np.linspace(cg.data_x_range_l,cg.data_x_range_u, N_points_domain).reshape((N_points_domain,1))

    # initialize one of our networks
    w, b = create_computation_graph_linear(n_in,n_out)

    # projection from input x to output y through computational graph
    y_range = computation_graph_linear(x_range,w, b_opt)

    # check how this computational_graph predicts at the inputs denote by our observed data X.
    y_pred = computation_graph_linear(x_data,w, b_opt)

    # compute the two losses at the predictions
    squared_loss = squared_loss_function(t_data, y_pred)

    # compute expected loss per parameter 
    expected_squared_loss_acc.append(np.sum(squared_loss))
    
    ## append for plotting
    w_range.append(np.squeeze(w))


# sort loss and weights to interactive plot later
idx = np.argsort(w_range)
sorted_w_range = np.array(w_range)[idx]
sorted_expected_squared_loss_acc = np.array(expected_squared_loss_acc)[idx]

## for plotting
w_opt = np.squeeze(w_opt)
b_opt = np.squeeze(b_opt)

## Plot optimal linear approximation and associated loss
fig = plt.figure(figsize = (10,10))
ax11 = fig.add_subplot(221)
ax12 = fig.add_subplot(222, projection='3d')
ax22 = fig.add_subplot(224)

## display data 
ax11.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

## Plot optimal funciton function
ax11.plot(x_data, y_pred_opt,'*', markersize = 5, color = f"C1", label = 'Predictions at training input data')
ax11.plot(x_range,y_range_opt, color = f"C1", label = 'function on all the domain' )

## set plot lims
ax11.set_xlabel('altura')
ax11.set_ylabel('peso')
ax11.set_title('Squared Euclidean Distance')
ax11.set_ylim([cg.losses_named['squared'].loss_y_lim_l,cg.losses_named['squared'].loss_y_lim_u])
ax11.set_xlim([cg.losses_named['squared'].loss_x_lim_l,cg.losses_named['squared'].loss_x_lim_u])

## Plot individual losses
for j in range(len(x_data)):
    ax11.text(x_data[j], -0.5, f"$d(y_{j+1},t_{j+1})={squared_loss_opt[j].item():.2}$", fontsize=8, ha="center", color = f"C1") 

ax11.text(0,5, f"$L(w = {w_opt:.2f},b = {b_opt:.2f})= {float(np.sum(squared_loss_opt)):.2f}$", color = f"C1")  
ax11.text(0,4.5, f"$y = {w_opt:.2f} \cdot x + {b_opt:.2f}$", color = f"k")  


## display loss functions
ax12.plot_surface(w_mesh, b_mesh, expected_squared_loss_mesh, cmap = 'gray', alpha = 0.75 )
ax12.set_xlabel('weight values')
ax12.set_ylabel('bias values')
ax12.set_zlabel('squared loss function')

## Plot the loss in the loss function view
ax12.scatter(w_opt, b_opt, expected_squared_loss_acc[i], color = f"C1")
ax12.text(w_opt, b_opt,expected_squared_loss_acc[i]+2.5, f"$L(w = {w_opt:.2f},b = {b_opt:.2f}) = {float(np.sum(squared_loss)):.2f}$", color = f"C1", zorder = 2, ha = 'center')    


## display loss functions fixing to optimal bias
ax22.plot(sorted_w_range, sorted_expected_squared_loss_acc )

ax22.plot(w_opt, float(np.sum(squared_loss_opt)), '*', color = f"C1")
ax22.text(w_opt,float(np.sum(squared_loss_opt)), f"$L(w = {w_opt:.2f},b = {b_opt:.2f}) = {float(np.sum(squared_loss_opt)):.2f}$", color = f"C1")    

ax22.set_xlabel('weight values')
ax22.set_ylabel('squared loss function')

As expected, the math matches the result. By the way, there are much simpler ways to arrive at these mathematical results. Why have I not chosen to place them here? Well, basically, I have derived this myself and decided to go this way. However, after checking whether my result was correct, there is a much simpler way to arrive at this result by solving the system of equations in a different way. 

First of all, when solving $\frac{\partial L(w,b)}{\partial b} = 0$ instead of solving for $w$, as I did, solve for $b$. The resulting expression is much simpler to modify using the tricks I showed to arrive at: $b = \mu_t -w\mu_x$.

Second, when solving  $\frac{\partial L(w,b)}{\partial w} = 0$, you need to replace the value of $b$ obtained in the step before, so instead of substituting $w$ as I did, substitute $b$.

Up to this point, you must be totally familiar with the result we have obtained. In particular, in the subject "Análisis de datos," you are told that the optimal line describing some data  is obtained by:

$$
\begin{split}
w = \frac{ \mathbb{COV}[x,t]}{ \mathbb{VAR}[x] }  \\
b = \mu_t -w\mu_x
\end{split}
$$

I know this because I have taught this subject in the past. On the other hand, in econometrics, you call this the ordinary least squares method. While in econometrics, you derive this from the Gauss-Markov theorem, in this subject, we will be deriving it from a totally different perspective. We have done it through the concept of a loss function, and we will be generalizing it through the lens of probabilistic machine learning.


#### Optimization: from a multivariate perspective viewpoint

It is rather convenient to express the previous problem through the lens of a multivariate function. Why? Because we will obtain a general expression that can be used in many situations to obtain the optimal model. Also, because, as we will see, expressing things through multivariate calculus simplifies the mathematical expressions in two ways. First, it removes summations with different indices. Second, we will see in other chapters that loss functions start getting a bit uglier than what we have seen so far. So breaking things as multivariate compositions of functions helps a lot.

First of all, note that this linear relationship:

$$
y = w \cdot x + b,
$$

can be compactly expressed as follows. First, the bias can be added to the vector of weights by adding a $1$ to $x$ and expressing everything as a dot product. In particular, if: $\xvec = [x,1]$ and $\wvec=[w,b]$, then:

$$
y = \xvect \wvec = w \cdot x + b
$$


However, we really want to consider a set of $N$ points. Thus, if we arrange all of our points into a matrix 

$$
\Xmat =
\begin{pmatrix}
x_{1}, & 1 \\
x_{2}, & 1 \\
\vdots & \vdots \\
x_{N}, & 1
\end{pmatrix}
$$


Thus, our model is now expressed compactly as:

$$
\yvec = \Xmat \wvec 
$$

where now $\yvec$ is the prediction at each point. The loss function is now given by:

$$
\begin{split}
L(\wvec,\Xmat,\tvec) &= \onevect(\tvec -\yvec )^2 \\
&= \onevect(\tvec -\Xmat\wvec )^2 \\
& = \sum^N_{n=1} (t_n - y_n)^2 \\
& = \sum^N_{n=1} (t_n -{\xvect}^{(n)}\wvec )^2 \\
& = \sum^N_{n=1} (t_n -xw - b )^2 
\end{split}
$$

where $\tvec$ is the vector with the training labels. As you can see, all these equalities are equivalent, but it is rather more convenient to express everything using multivariate functions. We can now take the derivative by applying the chain rule, obtaining Jacobians, and then multiplying them. To do so, let's break the above expression into the following composition. Note that, as we saw in the [critical point of functions notebook](../../math/optimization/theory/CriticalPoints.ipynb), we might have a function $f:\mathbb{R}^N \to \mathbb{R}$ that, when broken down into a composition, this composition is made up of functions of arbitrary input and output shapes
:

$$
\begin{align*}
\yvec &= \Xmat\wvec && \mathbb{R}^2 \to \mathbb{R}^N\\
\lvec &= (\tvec -\yvec )^2 && \mathbb{R}^N \to \mathbb{R}^N \quad\quad \text{element-wise function} \\
L &= \onevect \lvec && \mathbb{R}^N \to \mathbb{R}
\end{align*}
$$

where Jacobians from each of the elements in the composition are given by:

$$
\begin{split}
\Jac{\lvec}{L} &= \onevect \in \mathbb{R}^{1\times N}\\
\Jac{\yvec}{\lvec} &= -2\diag(\tvec -\yvec) \in \mathbb{R}^{N \times N}\\
\Jac{\wvec}{\yvec} &= \Xmat \in \mathbb{R}^{N \times 2} \\
\end{split}
$$

As we see, the shape (rows and columns) of the Jacobian matches what we expect from the input and output dimensionalities of the functions. Thus, the Jacobian of the loss function wrt $\wvec$ is given by:

$$
\begin{split}
\Jac{\wvec}{L}  = -2\onevect\diag(\tvec -\Xmat\wvec)\Xmat
\end{split}
$$

One can check that this is equivalent to the expressions we have obtained before, by substituting the corresponding elements in the gradient:

$$
\begin{split}
\frac{\partial L(w,b)}{\partial w} = -\sum_{n=1}^N 2\cdot(t_n - w\cdot x_n - b)x_n \\
\frac{\partial L(w,b)}{\partial b} = -\sum_{n=1}^N 2\cdot(t_n - w\cdot x_n - b) 
\end{split}
$$

However, this expression allows us to obtain a general form for the linear regression problem. First, since we are interested in the gradient, we need to transpose the Jacobian.

$$
\begin{split}
\pareT{\Jac{\wvec}{L}}  = \grad{\wvec}{L} = -2\Xmatt\diag(\tvec -\Xmat\wvec)\onevec
\end{split}
$$

Now we can solve for $\wvec$:

$$
\begin{split}
 -2\Xmatt\diag(\tvec -\Xmat\wvec)\onevec = 0\\
 \Xmatt (\tvec -\Xmat\wvec) = 0\\
 \Xmatt \tvec - \Xmatt \Xmat\wvec = 0\\
 \pareinv{\Xmatt \Xmat}\Xmatt \Xmat\wvec = \pareinv{\Xmatt \Xmat}\Xmatt \tvec \\
 \wvec_\text{opt} = \pareinv{\Xmatt \Xmat}\Xmatt \tvec
\end{split}
$$

You should recognize this as the general expression of the ordinary least squares method. But as you see, it is not different from what we are taught in high school. When you want the minimum of a function, obtain the derivative, set it to 0, and solve the equation or system of equations. 

Then, why do we name this the ordinary least squares method?. Well, it has a strong connection with linear algebra. In particular, linear models with a square function can be written down compactly using the square of the Euclidean norm as:

$$
\begin{split}
L(\wvec,\Xmat,\tvec) &= \onevect(\tvec -\Xmat\wvec )^2 \\
&= ||\tvec -\Xmat\wvec||_2^2
\end{split}
$$

This is a linear system. The least squares method is a method to obtain the solution to a system of equations when no exact solution is available: https://en.wikipedia.org/wiki/Linear_least_squares. As we will see, all of the generalized ordinary least squares, ordinary least squares, etc can be more elegantly obtained from the probabilistic perspective.

### Optimization: via gradient descent.

Obviously, one can apply gradient descent and coordinate optimization as well. While coordinate optimization is of no practical interest, in general, for these linear models, it turns out that gradient descent is a very important algorithm.

Gradient descent works by iteratively applying the formula:

$$
\begin{split}
\wvec^{(t+1)} = \wvec^{(t)} -\alpha \grad{\wvec}{} L(\wvec^{(t)},\Xmat,\tvec)   
\end{split}
$$

with $ \grad{\wvec}{} L(\wvec^{(t)},\Xmat,\tvec) = -2\Xmatt\diag(\tvec -\Xmat\wvec)\onevec$.

Importantly, one does not need to compute $-2\Xmatt\diag(\tvec -\Xmat\wvec)\onevec$ explicitly, but just implement the most efficient version. For instance, there is no need to construct the diagonal matrix $\diag(\tvec -\Xmat\wvec)$, but it is enough to note that $\diag(\tvec -\Xmat\wvec)\onevec$ is directly the column vector $\tvec -\Xmat\wvec$. This way of working out the gradient is at the core of automatic differentiation. The principle behind this operation is known as the vector-Jacobian product (although in this case I have done a kind of vector-gradient computation). 

Let's visualize gradient descent on this linear regression problem by fixing the bias as before. So we just run gradient descent on the weight $w$.

In [ ]:
## ========================== ##
## ==== Gradient Descent ==== ##
## ========================== ##
loss_type = 'squared'

if loss_type not in ['absolute', 'squared']:
    raise RuntimeError("Invalid loss type choose from absolute or square")

## ======================== ##
## Simulation configuration ##
## ======================== ##
fixed_bias = cg.fixed_bias

## fix seed so that randomness is controlled.
np.random.seed(cg.seed)

## number of points in the domain used to plot the functions 
N_points_domain = cg.N_domain_x

## get number of losses to display
num_losses = len(cg.losses)

## ================ ##
## For plot display ##
## ================ ##
## create figure box
fig = plt.figure(figsize=(10, 10))
gs = fig.add_gridspec(2, 2)

ax11 = fig.add_subplot(gs[0, 0])  
ax12 = fig.add_subplot(gs[0, 1])  
ax21 = fig.add_subplot(gs[1, 0])  

fig.subplots_adjust(wspace=0.5)

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

## =============================================
## Specify input output to the computation graph
D_in = 1
D_out = 1

## ============================================
## Get the loss function over which we optimize
w_range = np.linspace(cg.losses_named[loss_type].w_range_l,cg.losses_named[loss_type].w_range_u,cg.N_models_simulation).reshape((cg.N_models_simulation,n_in,n_out))

## get predictions for each model
y_data = computation_graph_linear(x_data, w_range, fixed_bias)

## compute loss
loss_range = cg.losses_named[loss_type].loss_fun(t_data, y_data)

## accumulate loss per datapoint
loss_acc_range = np.sum(loss_range, axis = 1)

## squeeze and display
loss_acc_range = np.squeeze(loss_acc_range)
w_range = np.squeeze(w_range)

## Display different models sequentially, alongside its loss.

# display loss function
ax11.plot(w_range, loss_acc_range, color = 'C0')
ax11.set_xlabel('Weight')
ax11.set_ylabel('Loss')

# Initialize parameters
w = np.array([cg.losses_named[loss_type].w_init]).reshape(n_in,n_out)

## gradient descent parameters
lr = 0.05
epochs = 10
loss_acc_itet = []

for e in range(epochs):

    ## forward plus backward
    grad_w, _ = cg.losses_named[loss_type].grad_loss_fun(x_data,t_data, w, fixed_bias)
    
    ## compute function at current parameter value
    function = computation_graph_linear(x_range, w, fixed_bias)

    ## compute predictions at current parameter value
    y_data = computation_graph_linear(x_data, w, fixed_bias)

    ## compute loss at current parameter value
    loss = cg.losses_named[loss_type].loss_fun(t_data,y_data)    
    loss_acc = np.sum(loss)

    # save loss to show over the course of learning
    loss_acc_itet.append(loss_acc)

    ## get the gradient function at the point w (tangent at the point)
    gradient_function_w_at_current_w = grad_w * w_range + loss_acc - grad_w * w
    
    ## compute loss on updated parameters
    w_n = w-lr*grad_w
    
    ## function on new parameters
    function_n = computation_graph_linear(x_range, w_n, fixed_bias)
    
    ## predictions with new parameters
    y_data_n = computation_graph_linear(x_data, w_n, fixed_bias)

    ## compute loss at current parameter value
    loss_n = cg.losses_named[loss_type].loss_fun(t_data, y_data_n)
    loss_acc_n = np.sum(loss_n)
    
    ## ============= ##
    ## ============= ##
    ## START DRAWING ##
    ## ============= ##
    ## ============= ##
    # Clear previous data
    ax11.clear()
    ax12.clear()
    ax21.clear()
    
    w_plot = np.squeeze(w)
    w_plot_n = np.squeeze(w_n)
    grad_w_plot = np.squeeze(grad_w)
    x_data_plot = np.squeeze(x_data)
    t_data_plot = np.squeeze(t_data)
    y_data_plot = np.squeeze(y_data)
    y_data_plot_n = np.squeeze(y_data_n)
    loss_plot = np.squeeze(loss)
    loss_plot_n = np.squeeze(loss_n)

    ## ========================================= ##
    ## loss function over the course of learning ##
    ax21.plot(np.arange(e+1), np.array(loss_acc_itet),'o',color = 'C0')
    ax21.plot(np.arange(e+1), np.array(loss_acc_itet),color = 'C0')
    ax21.set_xlabel("Epochs")
    ax21.set_ylabel("Loss")
    ax21.set_xlim([0,epochs])
    

    ## =========================== ##
    ## prediction function picture ##
    ax12.plot(x_range,function, color = 'C1', label = 'function: y = w*x')
    ax12.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

    ## plot loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot,loss_plot)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C1', label = 'network prediction')
        else:
            ax12.plot(xi,yi, 'x', color = 'C1')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C1", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C1" ) 

    # label function with the weight at that moment
    ax12.text(x_range[-20],function[-20], f'w = {w_plot:.2f}', color = 'k', fontsize = 12)
    
    ax12.text(-5, 33, f"Iteration {e}, {loss_type} loss = {loss_acc:.2f}", fontsize=12, va='bottom', color = f"C1" ) 
    ax12.set_xlabel(cg.data_x_name)
    ax12.set_ylabel(cg.data_y_name)
    ax12.set_ylim([cg.data_y_lim_l_gd_pred_fun,cg.data_y_lim_u_gd_pred_fun])
    ax12.legend()
    
    ## ===================== ##
    ## loss function picture ##
    ## 0. label and axis limits
    ax11.set_xlabel('Weight')
    ax11.set_ylabel('Loss')
    ax11.set_ylim([cg.losses_named[loss_type].gd_loss_fun_y_lim_l,cg.losses_named[loss_type].gd_loss_fun_y_lim_u])
    ax11.set_xlim([cg.losses_named[loss_type].gd_loss_fun_x_lim_l,cg.losses_named[loss_type].gd_loss_fun_x_lim_u])
          
    ## 1. display loss function
    ax11.plot(w_range, loss_acc_range, color = 'C0', label = 'loss', zorder = 20)    
    
    ## 2. display current weight
    ax11.plot(w_plot, cg.losses_named[loss_type].gd_loss_fun_display_param_y + cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y, '*', color = 'C1', label = 'current weight', zorder = 50, markersize = 10)
    ax11.text(w_plot + 1, cg.losses_named[loss_type].gd_loss_fun_display_param_y + cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y , f"w = {w_plot:.2f}", fontsize=12, va='bottom', color = f"C1" , zorder = 50)
    ax11.legend()    
        
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## animation by drawing horizontal lines on current parameter and updated parameter values
    ax11.vlines(np.squeeze(w), ymin=cg.losses_named[loss_type].gd_loss_fun_display_param_y, ymax=loss_acc, color='k', linestyles='dotted', zorder = -50)

    ## 3. display current loss
    ax11.plot(w_plot, loss_acc, 'o', color = 'C0', label = 'loss at current weight', zorder = 20)
    ax11.text(w_plot + 0.5, loss_acc , f"loss = {loss_acc:.2f}", fontsize=12, va='bottom', color = "C0" , zorder = 50)
    ax11.legend()
    
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## 4. display the gradient function
    ax11.plot(w_range, np.squeeze(gradient_function_w_at_current_w), color = 'C2', label = 'gradient function: f(w) = grad_w * w + loss - grad_w * w', zorder = 20)
    ax11.text(w_range[-1], np.squeeze(gradient_function_w_at_current_w)[-1], f"grad_w = {grad_w_plot:.2f}", fontsize=12, va='bottom', color = f"C2" , zorder = 200) 
    ax11.legend(loc = 'lower right')
    
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## draw rest of lines to show update
    ax11.hlines(y = loss_acc, xmin=w_plot_n, xmax=w_plot, color='k', linestyles='dotted', zorder = -50)
    
    writer.append_data(frame)
    
    ax11.vlines(w_plot_n, ymin=cg.losses_named[loss_type].gd_loss_fun_display_param_y , ymax=loss_acc, color='k', linestyles='dotted', zorder = -50)
    
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

    ## 5. display new weight
    ax11.plot(w_plot_n, cg.losses_named[loss_type].gd_loss_fun_display_param_y + cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y, '*', color = 'C3', label = 'updated weight: w_new = w - lr*grad_w', zorder = 200, markersize = 10)
    ax11.text(w_plot_n, cg.losses_named[loss_type].gd_loss_fun_display_param_y + 20*cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y, f"w_new = {w_plot:.2f} -{lr:.2f}*{grad_w_plot:.2f} = {w_plot-lr*grad_w_plot:.2f}", fontsize=12, va='bottom', color = f"C3" , zorder = 200) 
    ax11.legend()
        
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## 6. display updated function
    ax12.plot(x_range,function_n, color = 'C3')

    ## plot squared loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot_n,loss_plot_n)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C3')
        else:
            ax12.plot(xi,yi, 'x', color = 'C3')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C3", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C3" ) 

    # label function with the weight at that moment
    ax12.text(x_range[5],function_n[5], f'w = {w_plot_n:.2f}; b = {fixed_bias:.2f}', color = 'C3', fontsize = 12)
    
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## update parameter with gradient descent, for the next update
    w = w-lr*grad_w
    
writer.close() 
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

And we can obviously run it over the vector $\wvec$.

In [ ]:
## ======================================================================================= ##
## display loss as a function of weight and bias parameter (loss incurred by each network) ##
## ======================================================================================= ##
## Let's see the associated loss to each possible function but seeing the loss
## as a function of the weight and bias parameter. 
## We show two different losses: squared (top) and absolute ( bottom )
loss_type = 'squared'

if loss_type not in ['absolute', 'squared']:
    raise RuntimeError("Invalid loss type choose from absolute or square")

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")    
    
## create figure box
fig = plt.figure(figsize=(10, 10))
gs = fig.add_gridspec(2, 2)

ax11 =  fig.add_subplot(gs[0 , 0], projection='3d')
ax12 = fig.add_subplot(gs[0, 1])  
ax21 = fig.add_subplot(gs[1, 0])  

fig.subplots_adjust(wspace=0.5)

## ===========================================
## Neural network specification for each layer

# neurons of input layer
n_in = 1
# neurons of output layer
n_out = 1

## ================================================================================
## Create several possible functions that our specific neural network can implement
## first of all draw loss function against a set of parameters

## To do so we need a mesh
w_mesh, b_mesh = np.meshgrid(
    np.linspace(cg.losses_named[loss_type].w_range_l_2d,cg.losses_named[loss_type].w_range_u_2d,cg.N_models_simulation),
    np.linspace(cg.losses_named[loss_type].b_range_l_2d,cg.losses_named[loss_type].b_range_u_2d,cg.N_models_simulation)
)

# reshape x_data and t_data for computations. t_data uses broadcasting.
x_data_expanded = x_data[:,np.newaxis]
t_data_expanded = t_data[:,np.newaxis]

# compute linear projection at all pairs of points
y_data_expanded = w_mesh*x_data_expanded + b_mesh

# compute loss
loss_acc_mesh = np.sum(cg.losses_named[loss_type].loss_fun(t_data_expanded, y_data_expanded), axis = 0)

ax11.plot_surface(w_mesh, b_mesh, loss_acc_mesh, cmap = 'gray')
ax11.set_xlabel('weight values')
ax11.set_ylabel('bias values')
ax11.set_zlabel(f'{loss_type} loss function')

# Initialize parameters
w = np.array([cg.losses_named[loss_type].w_init_full]).reshape(n_in,n_out)
b = np.array([cg.losses_named[loss_type].b_init_full])

## gradient descent parameters
lr = 0.01 # try 0.1, 0.01, 0.15, 0.21 to show: fast convergence, slow convergence, convergence with bumping, divergence
# lr absolute = 0.5
epochs = 20
loss_acc_itet = []

for e in range(epochs):

    ## forward plus backward
    grad_w, grad_b = cg.losses_named[loss_type].grad_loss_fun(x_data,t_data, w, b)

    ## compute function at current parameter value
    function = computation_graph_linear(x_range, w, b)

    ## compute predictions at current parameter value
    y_data = computation_graph_linear(x_data, w, b)

    ## compute loss at current parameter value
    loss = cg.losses_named[loss_type].loss_fun(t_data, y_data)
    loss_acc = np.sum(loss)
    loss_acc_itet.append(loss_acc)
    
    ## compute loss on updated parameters
    w_n = w-lr*grad_w
    b_n = b-lr*grad_b
    
    ## function on new parameters
    function_n = computation_graph_linear(x_range, w_n, b_n)
    
    ## predictions with new parameters
    y_data_n = computation_graph_linear(x_data, w_n, b_n)

    ## compute loss at current parameter value
    loss_n = cg.losses_named[loss_type].loss_fun(t_data, y_data_n)

    loss_acc_n = np.sum(loss_n)
    
    ## ============= ##
    ## ============= ##
    ## START DRAWING ##
    ## ============= ##
    ## ============= ##
    # Clear previous data
    ax11.clear()
    ax12.clear()
    ax21.clear()
    
    w_plot = np.squeeze(w)
    b_plot = np.squeeze(b)
    
    w_plot_n = np.squeeze(w_n)
    b_plot_n = np.squeeze(b_n)

    x_data_plot = np.squeeze(x_data)
    t_data_plot = np.squeeze(t_data)
    y_data_plot = np.squeeze(y_data)
    y_data_plot_n = np.squeeze(y_data_n)
    loss_plot = np.squeeze(loss)
    loss_plot_n = np.squeeze(loss_n)

    ## ========================================= ##
    ## loss function over the course of learning ##
    ax21.plot(np.arange(e+1), np.array(loss_acc_itet),'o',color = 'C0')
    ax21.plot(np.arange(e+1), np.array(loss_acc_itet),color = 'C0')
    ax21.set_xlabel("Epochs")
    ax21.set_ylabel("Loss")
    ax21.set_xlim([0,epochs])
    
    ## ================ ##
    ## function picture ##
    ax12.plot(x_range,function, color = 'C1', label = 'function: y = w*x + b')
    ax12.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

    ## plot squared loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot,loss_plot)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C1', label = 'network prediction')
        else:
            ax12.plot(xi,yi, 'x', color = 'C1')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C1", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C1" ) 

    # label function with the weight at that moment
    ax12.text(x_range[-20],function[-20], f'w = {w_plot:.2f}; b = {b_plot:.2f}', color = 'C1', fontsize = 12)
    
    ax12.text(-5, 33, f"Iteration {e}, {cg.losses_named[loss_type].loss_name} = {loss_acc:.2f}", fontsize=12, va='bottom', color = f"C1" ) 
    ax12.set_xlabel(cg.data_x_name)
    ax12.set_ylabel(cg.data_y_name)
    ax12.set_ylim([cg.data_y_lim_l_gd_pred_fun,cg.data_y_lim_u_gd_pred_fun])
    ax12.legend()
    
    ## ===================== ##
    ## loss function picture ##
    
    ## 1. display loss function
    ax11.plot_surface(w_mesh, b_mesh, loss_acc_mesh, cmap = 'gray')
    ax11.set_xlabel('weight values')
    ax11.set_ylabel('bias values')
    ax11.set_zlabel(f'{cg.losses_named[loss_type].loss_name} loss function')
    
    ## 2. display current weight
    ax11.plot(w_plot, b_plot, loss_acc, 'o', color = 'C1', label = 'current weight', zorder = 50, markersize = 5)
    ax11.text(w_plot, b_plot, loss_acc, f"(w,b) = ({w_plot:.2f},{b_plot:.2f})", fontsize=12, va='bottom', color = f"C1" , zorder = 50)
    ax11.legend()
    
    ## save image frame
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## 3. display the gradient arrow and updated weight
    a = Arrow3D([w_plot, w_plot_n], [b_plot, b_plot_n], [loss_acc, loss_acc_n],
                mutation_scale=20, lw=0.5, arrowstyle="-|>", color="C0")
    ax11.add_artist(a)
    ax11.legend()

    ## 4. display new weight
    ax11.plot(w_plot_n, b_plot_n, loss_acc_n, 'o', color = 'C3', label = 'updated weight', zorder = 50, markersize = 5)
    ax11.text(w_plot_n, b_plot_n, loss_acc_n, f"(w_new,b_new) = ({w_plot_n:.2f},{b_plot_n:.2f})", fontsize=12, va='bottom', color = f"C3" , zorder = 50)
    ax11.legend()
    
    ## save image frame
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## 6. display updated function
    ax12.plot(x_range,function_n, color = 'C3')
    ax12.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

    ## plot squared loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot_n,loss_plot_n)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C3')
        else:
            ax12.plot(xi,yi, 'x', color = 'C3')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C3", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C3" ) 

    # label function with the weight at that moment
    ax12.text(x_range[5],function_n[5], f'w = {w_plot_n:.2f}; b = {b_plot_n:.2f}', color = 'C3', fontsize = 12)
    
    ## save image frame
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## update parameter with gradient descent, for the next update
    w = w-lr*grad_w
    b = b-lr*grad_b
    
writer.close() 
plt.close()


In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

## Absolute Loss

We have seen how to obtain the best linear model wrt the squared loss. It turns out that we can do the same with the absolute loss. As we will see, the solution to the problem, ie, the optimal line, changes.

With the absolute loss, the ordinary least squares method does not apply, because there is no analytical way in which we solve the system that results from equating the gradient to zero.

We can obtain the gradient of the absolute loss, again, by using the chain rule. In this case, the composition is given by:

$$
\begin{split}
\yvec & = \Xmat\wvec\\
\lvec & = |\tvec -\yvec|  \quad \text{element-wise function} \\
L & = \onevect \lvec\\
\end{split}
$$

The problem here is that the absolute function $|x|$ is non-differentiable at $0$. Here we usually use the subgradient. It is common to select one of the elements of the subgradient. Here we will select the value of $0$. This means that when $x=0$, the derivative at this point is zero.

where Jacobians from each of the elements in the composition are given by:

$$
\begin{split}
\Jac{\lvec}{L} &= \onevect\\
\Jac{\yvec}{\lvec} &= -\diag\left(\begin{cases}
1, & \tvec -\yvec  > 0,\\
-1, & \tvec -\yvec < 0\\
0, & \tvec -\yvec = 0
\end{cases}\right)\\
\Jac{\wvec}{\yvec} &= \Xmat\\
\end{split}
$$

Thus, the Jacobian of the loss function wrt $\wvec$ is given by:

$$
\begin{split}
\Jac{\wvec}{L} = -\onevect \Jac{\yvec}{\lvec}\Xmat
\end{split}
$$

We can optimize using gradient descent, again first by fixing the bias to a given value and then over the vector $\wvec$.

In [ ]:
## ========================== ##
## ==== Gradient Descent ==== ##
## ========================== ##
loss_type = 'absolute'

if loss_type not in ['absolute', 'squared']:
    raise RuntimeError("Invalid loss type choose from absolute or square")

## ======================== ##
## Simulation configuration ##
## ======================== ##
fixed_bias = cg.fixed_bias

## fix seed so that randomness is controlled.
np.random.seed(cg.seed)

## number of points in the domain used to plot the functions 
N_points_domain = cg.N_domain_x

## get number of losses to display
num_losses = len(cg.losses)

## ================ ##
## For plot display ##
## ================ ##
## create figure box
fig = plt.figure(figsize=(10, 10))
gs = fig.add_gridspec(2, 2)

ax11 = fig.add_subplot(gs[0, 0])  
ax12 = fig.add_subplot(gs[0, 1])  
ax21 = fig.add_subplot(gs[1, 0])  

fig.subplots_adjust(wspace=0.5)

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")

## =============================================
## Specify input output to the computation graph
D_in = 1
D_out = 1

## ============================================
## Get the loss function over which we optimize
w_range = np.linspace(cg.losses_named[loss_type].w_range_l,cg.losses_named[loss_type].w_range_u,cg.N_models_simulation).reshape((cg.N_models_simulation,n_in,n_out))

## get predictions for each model
y_data = computation_graph_linear(x_data, w_range, fixed_bias)

## compute loss
loss_range = cg.losses_named[loss_type].loss_fun(t_data, y_data)

## accumulate loss per datapoint
loss_acc_range = np.sum(loss_range, axis = 1)

## squeeze and display
loss_acc_range = np.squeeze(loss_acc_range)
w_range = np.squeeze(w_range)

## Display different models sequentially, alongside its loss.

# display loss function
ax11.plot(w_range, loss_acc_range, color = 'C0')
ax11.set_xlabel('Weight')
ax11.set_ylabel('Loss')

# Initialize parameters
w = np.array([cg.losses_named[loss_type].w_init]).reshape(n_in,n_out)

## gradient descent parameters
lr = 0.05
epochs = 10
loss_acc_itet = []

for e in range(epochs):

    ## forward plus backward
    grad_w, _ = cg.losses_named[loss_type].grad_loss_fun(x_data,t_data, w, fixed_bias)
    
    ## compute function at current parameter value
    function = computation_graph_linear(x_range, w, fixed_bias)

    ## compute predictions at current parameter value
    y_data = computation_graph_linear(x_data, w, fixed_bias)

    ## compute loss at current parameter value
    loss = cg.losses_named[loss_type].loss_fun(t_data,y_data)    
    loss_acc = np.sum(loss)

    # save loss to show over the course of learning
    loss_acc_itet.append(loss_acc)

    ## get the gradient function at the point w (tangent at the point)
    gradient_function_w_at_current_w = grad_w * w_range + loss_acc - grad_w * w
    
    ## compute loss on updated parameters
    w_n = w-lr*grad_w
    
    ## function on new parameters
    function_n = computation_graph_linear(x_range, w_n, fixed_bias)
    
    ## predictions with new parameters
    y_data_n = computation_graph_linear(x_data, w_n, fixed_bias)

    ## compute loss at current parameter value
    loss_n = cg.losses_named[loss_type].loss_fun(t_data, y_data_n)
    loss_acc_n = np.sum(loss_n)
    
    ## ============= ##
    ## ============= ##
    ## START DRAWING ##
    ## ============= ##
    ## ============= ##
    # Clear previous data
    ax11.clear()
    ax12.clear()
    ax21.clear()
    
    w_plot = np.squeeze(w)
    w_plot_n = np.squeeze(w_n)
    grad_w_plot = np.squeeze(grad_w)
    x_data_plot = np.squeeze(x_data)
    t_data_plot = np.squeeze(t_data)
    y_data_plot = np.squeeze(y_data)
    y_data_plot_n = np.squeeze(y_data_n)
    loss_plot = np.squeeze(loss)
    loss_plot_n = np.squeeze(loss_n)

    ## ========================================= ##
    ## loss function over the course of learning ##
    ax21.plot(np.arange(e+1), np.array(loss_acc_itet),'o',color = 'C0')
    ax21.plot(np.arange(e+1), np.array(loss_acc_itet),color = 'C0')
    ax21.set_xlabel("Epochs")
    ax21.set_ylabel("Loss")
    ax21.set_xlim([0,epochs])
    

    ## =========================== ##
    ## prediction function picture ##
    ax12.plot(x_range,function, color = 'C1', label = 'function: y = w*x')
    ax12.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

    ## plot loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot,loss_plot)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C1', label = 'network prediction')
        else:
            ax12.plot(xi,yi, 'x', color = 'C1')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C1", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C1" ) 

    # label function with the weight at that moment
    ax12.text(x_range[-20],function[-20], f'w = {w_plot:.2f}', color = 'k', fontsize = 12)
    
    ax12.text(-5, 13, f"Iteration {e}, {loss_type} loss = {loss_acc:.2f}", fontsize=12, va='bottom', color = f"C1" ) 
    ax12.set_xlabel(cg.data_x_name)
    ax12.set_ylabel(cg.data_y_name)
    ax12.set_ylim([cg.data_y_lim_l_gd_pred_fun,cg.data_y_lim_u_gd_pred_fun])
    ax12.legend()
    
    ## ===================== ##
    ## loss function picture ##
    ## 0. label and axis limits
    ax11.set_xlabel('Weight')
    ax11.set_ylabel('Loss')
    ax11.set_ylim([cg.losses_named[loss_type].gd_loss_fun_y_lim_l,cg.losses_named[loss_type].gd_loss_fun_y_lim_u])
    ax11.set_xlim([cg.losses_named[loss_type].gd_loss_fun_x_lim_l,cg.losses_named[loss_type].gd_loss_fun_x_lim_u])
          
    ## 1. display loss function
    ax11.plot(w_range, loss_acc_range, color = 'C0', label = 'loss', zorder = 20)    
    
    ## 2. display current weight
    ax11.plot(w_plot, cg.losses_named[loss_type].gd_loss_fun_display_param_y + cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y, '*', color = 'C1', label = 'current weight', zorder = 50, markersize = 10)
    ax11.text(w_plot + 1, cg.losses_named[loss_type].gd_loss_fun_display_param_y + cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y , f"w = {w_plot:.2f}", fontsize=12, va='bottom', color = f"C1" , zorder = 50)
    ax11.legend()    
        
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## animation by drawing horizontal lines on current parameter and updated parameter values
    ax11.vlines(np.squeeze(w), ymin=cg.losses_named[loss_type].gd_loss_fun_display_param_y, ymax=loss_acc, color='k', linestyles='dotted', zorder = -50)

    ## 3. display current loss
    ax11.plot(w_plot, loss_acc, 'o', color = 'C0', label = 'loss at current weight', zorder = 20)
    ax11.text(w_plot + 0.5, loss_acc , f"loss = {loss_acc:.2f}", fontsize=12, va='bottom', color = "C0" , zorder = 50)
    ax11.legend()
    
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## 4. display the gradient function
    ax11.plot(w_range, np.squeeze(gradient_function_w_at_current_w), color = 'C2', label = 'gradient function: f(w) = grad_w * w + loss - grad_w * w', zorder = 20)
    ax11.text(w_range[-1], np.squeeze(gradient_function_w_at_current_w)[-1], f"grad_w = {grad_w_plot:.2f}", fontsize=12, va='bottom', color = f"C2" , zorder = 200) 
    ax11.legend(loc = 'lower right')
    
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## draw rest of lines to show update
    ax11.hlines(y = loss_acc, xmin=w_plot_n, xmax=w_plot, color='k', linestyles='dotted', zorder = -50)
    
    writer.append_data(frame)
    
    ax11.vlines(w_plot_n, ymin=cg.losses_named[loss_type].gd_loss_fun_display_param_y , ymax=loss_acc, color='k', linestyles='dotted', zorder = -50)
    
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

    ## 5. display new weight
    ax11.plot(w_plot_n, cg.losses_named[loss_type].gd_loss_fun_display_param_y + cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y, '*', color = 'C3', label = 'updated weight: w_new = w - lr*grad_w', zorder = 200, markersize = 10)
    ax11.text(w_plot_n, cg.losses_named[loss_type].gd_loss_fun_display_param_y + 20*cg.losses_named[loss_type].gd_loss_fun_display_param_inc_y, f"w_new = {w_plot:.2f} -{lr:.2f}*{grad_w_plot:.2f} = {w_plot-lr*grad_w_plot:.2f}", fontsize=12, va='bottom', color = f"C3" , zorder = 200) 
    ax11.legend()
        
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## 6. display updated function
    ax12.plot(x_range,function_n, color = 'C3')

    ## plot squared loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot_n,loss_plot_n)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C3')
        else:
            ax12.plot(xi,yi, 'x', color = 'C3')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C3", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C3" ) 

    # label function with the weight at that moment
    ax12.text(x_range[5],function_n[5], f'w = {w_plot_n:.2f}; b = {fixed_bias:.2f}', color = 'C3', fontsize = 12)
    
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## update parameter with gradient descent, for the next update
    w = w-lr*grad_w
    
writer.close() 
plt.close()

In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

In [ ]:
## ======================================================================================= ##
## display loss as a function of weight and bias parameter (loss incurred by each network) ##
## ======================================================================================= ##
## Let's see the associated loss to each possible function but seeing the loss
## as a function of the weight and bias parameter. 
## We show two different losses: squared (top) and absolute ( bottom )
loss_type = 'absolute'

if loss_type not in ['absolute', 'squared']:
    raise RuntimeError("Invalid loss type choose from absolute or square")

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")    
    
## create figure box
fig = plt.figure(figsize=(10, 10))
gs = fig.add_gridspec(2, 2)

ax11 =  fig.add_subplot(gs[0 , 0], projection='3d')
ax12 = fig.add_subplot(gs[0, 1])  
ax21 = fig.add_subplot(gs[1, 0])  

fig.subplots_adjust(wspace=0.5)

## ===========================================
## Neural network specification for each layer

# neurons of input layer
n_in = 1
# neurons of output layer
n_out = 1

## ================================================================================
## Create several possible functions that our specific neural network can implement
## first of all draw loss function against a set of parameters

## To do so we need a mesh
w_mesh, b_mesh = np.meshgrid(
    np.linspace(cg.losses_named[loss_type].w_range_l_2d,cg.losses_named[loss_type].w_range_u_2d,cg.N_models_simulation),
    np.linspace(cg.losses_named[loss_type].b_range_l_2d,cg.losses_named[loss_type].b_range_u_2d,cg.N_models_simulation)
)

# reshape x_data and t_data for computations. t_data uses broadcasting.
x_data_expanded = x_data[:,np.newaxis]
t_data_expanded = t_data[:,np.newaxis]

# compute linear projection at all pairs of points
y_data_expanded = w_mesh*x_data_expanded + b_mesh

# compute loss
loss_acc_mesh = np.sum(cg.losses_named[loss_type].loss_fun(t_data_expanded, y_data_expanded), axis = 0)

ax11.plot_surface(w_mesh, b_mesh, loss_acc_mesh, cmap = 'gray')
ax11.set_xlabel('weight values')
ax11.set_ylabel('bias values')
ax11.set_zlabel(f'{loss_type} loss function')

# Initialize parameters
w = np.array([cg.losses_named[loss_type].w_init_full]).reshape(n_in,n_out)
b = np.array([cg.losses_named[loss_type].b_init_full])

## gradient descent parameters
lr = 0.5
epochs = 20
loss_acc_itet = []

for e in range(epochs):

    ## forward plus backward
    grad_w, grad_b = cg.losses_named[loss_type].grad_loss_fun(x_data,t_data, w, b)

    ## compute function at current parameter value
    function = computation_graph_linear(x_range, w, b)

    ## compute predictions at current parameter value
    y_data = computation_graph_linear(x_data, w, b)

    ## compute loss at current parameter value
    loss = cg.losses_named[loss_type].loss_fun(t_data, y_data)
    loss_acc = np.sum(loss)
    loss_acc_itet.append(loss_acc)
    
    ## compute loss on updated parameters
    w_n = w-lr*grad_w
    b_n = b-lr*grad_b
    
    ## function on new parameters
    function_n = computation_graph_linear(x_range, w_n, b_n)
    
    ## predictions with new parameters
    y_data_n = computation_graph_linear(x_data, w_n, b_n)

    ## compute loss at current parameter value
    loss_n = cg.losses_named[loss_type].loss_fun(t_data, y_data_n)

    loss_acc_n = np.sum(loss_n)
    
    ## ============= ##
    ## ============= ##
    ## START DRAWING ##
    ## ============= ##
    ## ============= ##
    # Clear previous data
    ax11.clear()
    ax12.clear()
    ax21.clear()
    
    w_plot = np.squeeze(w)
    b_plot = np.squeeze(b)
    
    w_plot_n = np.squeeze(w_n)
    b_plot_n = np.squeeze(b_n)

    x_data_plot = np.squeeze(x_data)
    t_data_plot = np.squeeze(t_data)
    y_data_plot = np.squeeze(y_data)
    y_data_plot_n = np.squeeze(y_data_n)
    loss_plot = np.squeeze(loss)
    loss_plot_n = np.squeeze(loss_n)

    ## ========================================= ##
    ## loss function over the course of learning ##
    ax21.plot(np.arange(e+1), np.array(loss_acc_itet),'o',color = 'C0')
    ax21.plot(np.arange(e+1), np.array(loss_acc_itet),color = 'C0')
    ax21.set_xlabel("Epochs")
    ax21.set_ylabel("Loss")
    ax21.set_xlim([0,epochs])
    
    ## ================ ##
    ## function picture ##
    ax12.plot(x_range,function, color = 'C1', label = 'function: y = w*x + b')
    ax12.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

    ## plot squared loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot,loss_plot)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C1', label = 'network prediction')
        else:
            ax12.plot(xi,yi, 'x', color = 'C1')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C1", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C1" ) 

    # label function with the weight at that moment
    ax12.text(x_range[-20],function[-20], f'w = {w_plot:.2f}; b = {b_plot:.2f}', color = 'C1', fontsize = 12)
    
    ax12.text(-5, 33, f"Iteration {e}, {cg.losses_named[loss_type].loss_name} = {loss_acc:.2f}", fontsize=12, va='bottom', color = f"C1" ) 
    ax12.set_xlabel(cg.data_x_name)
    ax12.set_ylabel(cg.data_y_name)
    ax12.set_ylim([cg.data_y_lim_l_gd_pred_fun,cg.data_y_lim_u_gd_pred_fun])
    ax12.legend()
    
    ## ===================== ##
    ## loss function picture ##
    
    ## 1. display loss function
    ax11.plot_surface(w_mesh, b_mesh, loss_acc_mesh, cmap = 'gray')
    ax11.set_xlabel('weight values')
    ax11.set_ylabel('bias values')
    ax11.set_zlabel(f'{cg.losses_named[loss_type].loss_name} loss function')
    
    ## 2. display current weight
    ax11.plot(w_plot, b_plot, loss_acc, 'o', color = 'C1', label = 'current weight', zorder = 50, markersize = 5)
    ax11.text(w_plot, b_plot, loss_acc, f"(w,b) = ({w_plot:.2f},{b_plot:.2f})", fontsize=12, va='bottom', color = f"C1" , zorder = 50)
    ax11.legend()
    
    ## save image frame
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## 3. display the gradient arrow and updated weight
    a = Arrow3D([w_plot, w_plot_n], [b_plot, b_plot_n], [loss_acc, loss_acc_n],
                mutation_scale=20, lw=0.5, arrowstyle="-|>", color="C0")
    ax11.add_artist(a)
    ax11.legend()

    ## 4. display new weight
    ax11.plot(w_plot_n, b_plot_n, loss_acc_n, 'o', color = 'C3', label = 'updated weight', zorder = 50, markersize = 5)
    ax11.text(w_plot_n, b_plot_n, loss_acc_n, f"(w_new,b_new) = ({w_plot_n:.2f},{b_plot_n:.2f})", fontsize=12, va='bottom', color = f"C3" , zorder = 50)
    ax11.legend()
    
    ## save image frame
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## 6. display updated function
    ax12.plot(x_range,function_n, color = 'C3')
    ax12.plot(x_data,t_data,'o', markersize = 8, label = 'data observations')

    ## plot squared loss associated at each point and draw line between dots to highliht what the loss measures
    for idx, (xi, ti, yi, sl) in enumerate(zip(x_data_plot,t_data_plot,y_data_plot_n,loss_plot_n)):
        if idx == 0:
            ax12.plot(xi,yi, 'x', color = 'C3')
        else:
            ax12.plot(xi,yi, 'x', color = 'C3')
        ax12.plot([xi,xi], [ti, yi], '--',color = f"C3", alpha = 0.5)
        ax12.text(xi, yi, f'{sl:.2f}', fontsize=12, va='top', color = f"C3" ) 

    # label function with the weight at that moment
    ax12.text(x_range[5],function_n[5], f'w = {w_plot_n:.2f}; b = {b_plot_n:.2f}', color = 'C3', fontsize = 12)
    
    ## save image frame
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)
    
    ## update parameter with gradient descent, for the next update
    w = w-lr*grad_w
    b = b-lr*grad_b
    
writer.close() 
plt.close()


In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

## Multivariate linear regresion : $f: \mathbb{R}^D \rightarrow \mathbb{R}$

Linear regression is, in fact, the process of learning to approximate functions using hyperplanes. In 2 dimensions, this is just the normal plane. To get a broad insight on what linear regression is, consider, for instance, that we now have two variables: altura and peso, to predict some other thing. The linear model in this case can be compactly expressed using the inner product as before, through:

$$
\begin{split}
\yvec = \xvect \wvec  = x_1 w_1 + x_2 w_2 + b 
\end{split}
$$

Let's visualize some virtual data:

### Multidimensional data

In [ ]:
x_data = np.array([
    [-2.0, -1.0],
    [ 2.0, -1.5],
    [-1.0,  2.0],
    [ 2.5,  2.0],
])

t_data = np.array([
    -2.0,
     6.5,
    -3.0,
     4.0,
])

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")

ax.scatter(
    x_data[:, 0],
    x_data[:, 1],
    t_data,
    s=80,
    color="tab:blue"
)


ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_zlabel(r"$t$")

### Possible Hyperplanes

In the same way as before, let's plot some possible plausible plane that explains this data. Each one, will have it loss associated. We can use any of the losses. Now, I will just use the squared loss.

In [ ]:
## figure
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")

# Create temporary file for video creation
video_filename = "/tmp/aux.mp4"

## video writer
writer = imageio.get_writer(video_filename, format="FFMPEG", mode="I", fps=1, codec="libx264")    

# to draw planes
X1, X2 = np.meshgrid(
    np.linspace(-4, 4, 20),
    np.linspace(-4, 4, 20)
)
# to be able to perform dot product.
X_grid = np.stack([X1.ravel(), X2.ravel()], axis=1) 

# dimensionality of input and output
D_in = 2
D_out = 1

# Muestra planos aleatorios
for _ in range(15):
    
    # initialize one of our networks
    w, b = create_computation_graph_linear(D_in,D_out)

    # projection from input x to output y through computational graph
    y_grid = computation_graph_linear(X_grid, w, b)
    
    # volver a la forma de la malla
    Y = y_grid.reshape(X1.shape)

    ## Draw
    ax.clear()

    ax.scatter(
        x_data[:, 0],
        x_data[:, 1],
        t_data,
        s=80,
        color="tab:blue"
    )

    ax.plot_surface(
        X1, X2, Y,
        alpha=0.8,
        color="tab:red",
        linewidth=0
    )

    ax.set_xlabel(r"$x_1$")
    ax.set_ylabel(r"$x_2$")
    ax.set_zlabel(r"$y$")

    ax.set_xlim([-4,4])
    ax.set_ylim([-4,4])
    ax.set_zlim([-6,6])
    ax.view_init(elev=20, azim=-60)

    ## save image frame
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    writer.append_data(frame)

    
writer.close() 
plt.close()


In [ ]:
# Mostrar el video en Jupyter Notebook
display(Video(data=video_filename, embed=True))
os.remove(video_filename)

### Optimal Hyperplanes

We can use exactly the same methods as before to find the optimal plane with respect to the loss function. The cool thing about having worked out the gradients before is that we can plug those equations into either gradient descent or ordinary least squares directly. Note that now:

$$
\Xmat =
\begin{pmatrix}
x_1^{1},& x_2^{1}, & 1 \\
x_1^{2},& x_2^{2}, & 1 \\
\vdots ,& \vdots , & \vdots \\
x_1^{N},& x_2^{N},  & 1 \\
\end{pmatrix}
$$

This is a matrix where each row contains each data point, and each column contains its dimension. The first column the first dimension, and so on. Now:

$$
\begin{split}
\wvect = [w_1, w_2, 1]
\end{split}
$$

With this, given the squared loss, we have:

$$
\begin{split}
\yvec &= \Xmat\wvec\\
L &= ||\tvec - \yvec||_2^2
\end{split}
$$

Thus, if the squared loss is used, then the optimal weight can be obtained through ordinary least squares since the function being optimized is exactly the same, with more dimensions added to the vector or matrix. If the absolute loss is used, then we can use gradient descent using a similar expression to the one already derived

The optimal plane is obtained by:

$$
\begin{split}
\wvec_\text{opt} = \pareinv{\Xmatt \Xmat}\Xmatt \tvec
\end{split}
$$

Let's plot the optimal plane.


In [ ]:
# concatenate ones
x_fit = np.concatenate(
    [x_data, np.ones((x_data.shape[0], 1))],
    axis=1
)

# convert to column vector
t_data = np.reshape(t_data,(t_data.shape[0],1))

w_opt = fit_norm2_least_square(x_fit,t_data)

## figure
fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")

# to draw planes
X1, X2 = np.meshgrid(
    np.linspace(-4, 4, 20),
    np.linspace(-4, 4, 20)
)
# to be able to perform dot product.
X_grid = np.concatenate(
    [np.stack([X1.ravel(), X2.ravel()], axis=1) , np.ones((400, 1))],
    axis=1
)

y_grid = X_grid @ w_opt

# volver a la forma de la malla
Y = y_grid.reshape(X1.shape)

## Draw
ax.clear()

ax.scatter(
    x_data[:, 0],
    x_data[:, 1],
    t_data[:,0],
    s=80,
    color="tab:blue"
)

ax.plot_surface(
    X1, X2, Y,
    alpha=0.8,
    color="tab:red",
    linewidth=0
)

ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$x_2$")
ax.set_zlabel(r"$y$")

ax.set_xlim([-4,4])
ax.set_ylim([-4,4])
ax.set_zlim([-6,6])
ax.view_init(elev=20, azim=60)
ax.set_title("Optimal hyperplane")

We can obviously run gradient descent and plot the evolution of the different hyperplanes across the course of learning.

## Multioutput linear regression : $f: \mathbb{R} \rightarrow \mathbb{R}^C$

This is another type of linear model in which we want to predict multiple outputs from a single input. For example, we might want to predict the price of a house and the number of square meters from its location.

For a single point $x$ and two outputs, we have:

$$
\begin{split}
y_1 &= w_{11} x + b_1\\
y_2 &= w_{12} x + b_2\\
\end{split}
$$

This can be compactly expressed by noting that, when considering $N$ points and $C$ outputs, all the elements involved are now matrices.

$$
\begin{split}
\Ymat &= \Xmat\Wmat
\end{split}
$$


where:

$$
\begin{split}
\Xmat =
\begin{pmatrix}
x^{1},& 1 \\
x^{2},&  1 \\
\vdots ,& \vdots \\
x^{N},& 1 \\
\end{pmatrix}
\end{split}
$$


$$
\begin{split}
\Wmat =
\begin{pmatrix}
w_{11}, & w_{12}, & \dots & w_{1C}\\
b_{1} , & b_{2}, & \dots & b_{C}
\end{pmatrix}
\end{split}
$$


$$
\begin{split}
\Ymat =
\begin{pmatrix}
y_1^{1},& y_2^{1},  & \dots  & y_C^{1}\\
y_1^{2},&  y_2^{2}, & \dots & y_C^{2}\\
\vdots ,& \vdots,   & \vdots & \vdots \\
y_1^{N},& y_2^{N},  & \dots & y_C^{N} \\
\end{pmatrix}
\end{split}
$$


### Squared Loss function

As before, we can use the loss function we want depending on the problem. We will see later different aspects of different loss functions. Note that now our targets $t$ are not single scalars per data point, but vectors. This means that for $N$ points, $t$ will also be represented as a matrix. 

For multioutput models, there might be interest in considering dependencies between the different outputs. This can be achieved, for instance, by sharing parameters between different outputs. Also, it might be the case that the outputs present some form of dependency. We will see how the probabilistic perspective of linear regression offers a principled way to derive loss functions that consider our desired dependency. For the moment, we only consider independence.

Note that we just want to compare the square (or absolute) loss between the different pairs of outputs and sum over all data points. This is:

$$
\begin{split}
L(\Wmat,\Xmat,\Tmat) &= \sum^N_{n=1}\sum^C_{c=1}(t_c^n - y_c^n)^2\\
&=  \sum^N_{n=1}\left(\sum_{c=1}\left( t_c^n -   w_{1c} x^n - b_c\right)^2\right)
\end{split}
$$

Note that obtaining the gradient is not really complicated. At the core, we just have some summations and squares. However, one can start seeing that further multi-output models will start becoming more complicated. Here is where, again, expressing everything through multivariate operators helps a lot.  It turns out that the squared difference between the outputs in the elements between $ \Ymat$ and $ \Tmat$ can also be expressed through the square of a norm. For vectors, we had the square of the Euclidean norm. For matrices, we have the square of the Frobenius norm, which is equivalent to the trace of the transpose of a matrix and the matrix. This means:

$$
\begin{split}
L(\Wmat,\Xmat,\Tmat) &= \sum^N_{n=1}\sum^C_{c=1}(t_c^n - y_c^n)^2\\
&= ||\Tmat - \Ymat||^2_2\\
&= \tr{\pareT{\Tmat - \Ymat}\pare{\Tmat - \Ymat}}
\end{split}
$$


Thus, using multivariate compositions, the loss function can be compactly expressed as:

$$
\begin{split}
\Ymat = \Xmat \Wmat\\
\Cmat = \Tmat-\Ymat\\
L = \tr{\Cmatt\Cmat}
\end{split}
$$

Obtaining the Jacobian or gradient from this expression can be done as follows. We will use the chain rule and some rules from differential calculus. From the chain rule, we need to compute the Jacobians of each of the individual functions. The Jacobian from these expressions can be obtained by obtaining the first-order differential, expressing it in its canonical form, and then obtaining the Jacobian from it. Further information and references here: https://arxiv.org/pdf/2506.23996

The first-order differential of a scalar-valued matrix function can be written in two forms. I will use the one that uses vectorized expressions:

$$
\begin{split}
\dd y = \braT{\vvec \Amat} \dd \vvec \Xmat
\end{split}
$$

If we now work out the differential of the trace expression to yield the canonical form, we have (applying rules of differentials that can be found in the above reference and matrix tricks from the matrix cookbook):

$$
\begin{split}
\dd\bra{\tr{\Cmatt\Cmat}} &=  \tr{\dd[\Cmatt]\Cmat + \Cmatt\dd\Cmat}\\
&=\tr{\Cmatt\dd\Cmat + \Cmatt\dd\Cmat} \\
&=2\tr{\Cmatt\dd\Cmat} \\
&=2\braT{\vvec\Cmat}\dd\vvec\Cmat
\end{split}
$$

From this, the Jacobian is identified by $2\braT{\vvec\Cmat}$.

The Jacobian of the next expression is very simple to compute again using the rules of differentials. In this case have a matrix-valued matrix function, with canonical form given by:

$$
\begin{split}
\dd \vvec\Ymat = \Amat \dd \vvec \Xmat
\end{split}
$$

In our case, we have:

$$
\begin{split}
\dd\vvec[\Tmat - \Ymat] = -\dd\vvec\Ymat
\end{split}
$$

So here the Jacobian is given by $-\Imat$. Finally, we have a matrix-valued matrix function. The canonical form is the same as before, and so we have:

$$
\begin{split}
\dd\vvec\Ymat &= \dd\vvec[\Xmat\Wmat] \\
&= \vvec[\Xmat\dd\Wmat \Imat]\\
& =[\Imat \otimes \Xmat] \dd\vvec\Wmat
\end{split}
$$


from where the Jacobian is given by $[\Imat \otimes \Xmat]$. Now, multiplying all terms and applying the chain rule, we have:

$$
\begin{split}
\Jac{\vvec\Wmat}{} L(\Xmat,\Wmat,\Tmat) &= -2\braT{\vvec\bra{\Tmat - \Ymat}}\Imat[\Imat \otimes \Xmat] \\
&= - 2[\vvec[\Tmat - \Xmat\Wmat]]^T[\Imat \otimes \Xmat]
\end{split}
$$

Now, since we are interested in the gradient, we need to transpose the whole Jacobian, which implies:

$$
\begin{split}
\grad{\vvec\Wmat}{} L(\Xmat,\Wmat,\Tmat) &= \pareT{- 2\braT{\vvec\bra{\Tmat - \Xmat\Wmat}}[\Imat \otimes \Xmat]}\\
&= -2 [\Imat \otimes \Xmatt]\vvec[\Tmat - \Xmat\Wmat]
\end{split}
$$

Now set derivative to $0$ and solve for $\Wmat$ we have:

$$
\begin{split}
-2 [\Imat \otimes \Xmatt]\vvec[\Tmat - \Xmat\Wmat] &= 0\\
[\Imat \otimes \Xmatt]\vvec[\Tmat] - [\Imat \otimes \Xmatt]\vvec[\Xmat\Wmat\Imat] &= 0\\
[\Imat \otimes \Xmatt]\vvec[\Tmat] - [\Imat \otimes \Xmatt](\Imat\otimes\Xmat)\vvec[\Wmat] &= 0\\
[\Imat \otimes \Xmatt]\vvec[\Tmat] - [\Imat \otimes \Xmatt\Xmat]\vvec[\Wmat] &= 0\\
[\Imat \otimes \Xmatt]\vvec[\Tmat]  &= [\Imat \otimes \Xmatt\Xmat]\vvec[\Wmat]\\
\pareinv{[\Imat \otimes \Xmatt\Xmat]}[\Imat \otimes \Xmatt]\vvec[\Tmat]  &= \vvec[\Wmat]\\
[\Imat \otimes \pareinv{\Xmatt\Xmat}][\Imat \otimes \Xmatt]\vvec[\Tmat]  &= \vvec[\Wmat]\\
[\Imat \otimes \pareinv{\Xmatt\Xmat}\Xmat]\vvec[\Tmat]  &= \vvec[\Wmat]\\
\vvec[\pareinv{\Xmatt\Xmat}\Xmatt\Tmat\Imat]  &= \vvec[\Wmat]\\
\vvec[\pareinv{\Xmatt\Xmat}\Xmatt\Tmat]  &= \vvec[\Wmat]
\end{split}
$$

Undoing the vec operator on both sides yields the optimal regression matrix given by:

$$
\begin{split}
\Wmat_\text{opt}=\pareinv{\Xmatt\Xmat}\Xmatt\Tmat
\end{split}
$$

Computation of the inverse should be done by running linear solver systems using the Cholesky factorization (see [solving linear systems in practice](../../math/computation/theory/solving_linear_systems_in_practice.ipynb)), i.e never use functions that invert a matrix. 

Overall, this means that any model can be fitted using this equation, which has a computational cost dominated by $\mathcal{O}((N\times d)^3)$ due to inverse computation. As we will see, this includes a couple of models that might be of interest, because it holds for any model that is linear in the parameters, i.e., where predictions can be implemented using a dot product. These are the linear models of basis functions we mentioned at the beginning of this chapter.

### Absolute Loss Function

The absolute loss cannot be expressed compactly using an inner product as we have done with the trace. However, we can use our trick of expressing summations as dot products with the vector of ones and then take the derivative.

Now our loss is given by:

$$
\begin{split}
L(\Wmat,\Xmat,\Tmat) &= \sum^N_{n=1}\sum^C_{c=1}|t_c^n - y_c^n|
\end{split}
$$

We can now use the following composition:


$$
\begin{split}
\Ymat &= \Xmat \Wmat\\
\Zmat &= \Tmat-\Ymat \quad \text{element-wise}\\
\Cmat &= |\Zmat| \quad \text{element-wise}\\
L &= \sum^N_{n=1}\sum^C_{c=1} \Cmat_{n,c}
\end{split}
$$


Note that the last sum is a sum over columns and rows, which can be expressed as:

$$
\begin{split}
L = \onevect\Cmat\onevec
\end{split}
$$

So we can now work out the Jacobian of each transformation as follows. Again, here we will use "advanced" matrix differential calculus. You have the reference up there to learn about this in case you are interested. So we want to work out the Jacobian of the following composition:

$$
\begin{split}
\Ymat &= \Xmat \Wmat \quad \mathbb{R}^{N\times D} \to \mathbb{R}^{N\times C} \\
\Zmat &= \Tmat-\Ymat \quad \mathbb{R}^{N\times C} \to \mathbb{R}^{N\times C} \\
\Cmat &= |\Zmat| \quad \mathbb{R}^{N\times C} \to \mathbb{R}^{N\times C} \\
L &= \onevect\Cmat\onevec \quad \mathbb{R}^{N\times C} \to \mathbb{R}
\end{split}
$$

This involves matrix-valued matrix-argument functions and a scalar-valued matrix-argument function (the last one). The canonical form for the first-order differential equation is given by:

$$
\begin{split}
\dd y = \Amat \dd \vvec \Xmat \\
\dd \vvec \Ymat = \Amat \dd \vvec \Xmat
\end{split}
$$

Thus, we have:

$$
\begin{split}
\dd L &= \dd\bra{\onevect\Cmat\onevec}\\
&= \tr{\onevec\onevect\dd\Cmat}\\
&= \vvec\braT{\onevec\onevect}\dd\vvec\Cmat
\end{split}
$$

from where the Jacobian $\Jac{\vvec\Cmat}{L} = \vvec\braT{\onevec\onevect}$. Now, for the remaining operations, we start with $\Cmat = |\Zmat|$. Note that while we previously worked out this Jacobian directly by inspection, I am going to show another way. Note that we could still do it by inspection. The idea relies on expressing the absolute value through an operation for which there are well-established identities. It turns out that Minka [https://tminka.github.io/papers/matrix/minka-matrix.pdf](https://tminka.github.io/papers/matrix/minka-matrix.pdf) provides nice identities about the element-wise (also known as the Hadamard product). Note that we can write the absolute value as:

$$
\begin{split}
\Cmat = |\Zmat| = \sgn\Zmat \circ \Zmat
\end{split}
$$

where $\sgn\Zmat$ is the sign function applied over $\Zmat$.  So it is a matrix made up of 1 and -1. Now, applying an identity from Minka, we have:

$$
\begin{split}
\dd \Cmat &= \dd \bra{\sgn\Zmat \circ \Zmat}\\
&= \dd \pare{\diag\bra{\vvec \sgn\Zmat } \vvec \Zmat}\\
&=  \diag\bra{\vvec \sgn\Zmat } \dd\vvec \Zmat
\end{split}
$$

from where the Jacobian is identified by: $\diag\bra{\vvec \sgn\Zmat }$. The operator diag takes as input a vector and returns a diagonal matrix with the diagonal given by this vector. Now we have to work out the differential for:

$$
\begin{split}
\dd \vvec \Zmat &= \dd \vvec \bra{\Tmat-\Ymat}\\
&=  - \dd \vvec \Ymat
\end{split}
$$

from which the Jacobian is identified by $-\Imat$. Finally, we have:

$$
\begin{split}
\dd \vvec \Ymat &= \dd\vvec\bra{\Xmat \Wmat}\\
&= \vvec\bra{\Xmat \dd\Wmat \Imat}\\
&= \bra{\Imat \otimes \Xmat}\dd \vvec \Wmat 
\end{split}
$$


So the overall Jacobian is obtained by multiplying individual Jacobians, which yields:

$$
\begin{split}
\Jac{\vvec\Wmat}{L} &= - \vvec\braT{\onevec\onevect}\diag\bra{\vvec \sgn\Zmat } \bra{\Imat \otimes \Xmat}\\
&= - \vvec\braT{\onevec\onevect}\diag\bra{\vvec \sgn \pare{\Tmat - \Xmat\Wmat} } \bra{\Imat \otimes \Xmat}
\end{split}
$$

which naturally yields a vector since we are operating on a vectorized matrix. This is not really a problem, since undoing the vec operator just requires rearranging columns. The final step would be to work out the most efficient way to compute the Jacobian. For instance, there is no need to create a diagonal matrix in memory. And then, from this simplified expression, see how it is expressed in unvectorized form. This is at the core of automatic differentiation software. The gradient would be the transpose of this expression, which is left as an exercise.

To express the Jacobian in unvectorized form we can make use of the identity $\braT{\vvec\bra{\Amat\Xmat\Bmat}}=\braT{\vvec \Xmat}\pare{\Bmat \otimes \Amatt}$. First using the identity $\pareT{\vvec\pare{\Amat \circ \Bmat}} = \braT{\vvec \Bmat} \diag\pare{\vvec(\Amat)}$:

$$
\begin{split}
\Jac{\vvec\Wmat}{L} &= - \braT{ \vvec\bra{ \onevec\onevect \circ \sgn \pare{\Tmat - \Xmat\Wmat} }} \bra{\Imat \otimes \Xmat}\\
&= -\braT{\vvec\bra{\Xmatt\bra{ \onevec\onevect \circ \sgn \pare{\Tmat - \Xmat\Wmat} }}}
\end{split}
$$

Undoing vec yields:

$$
\begin{split}
\Jac{\Wmat}{L} = -\Xmatt\bra{ \onevec\onevect \circ \sgn \pare{\Tmat - \Xmat\Wmat} }
\end{split}
$$

which yields a matrix.


## Multioutput linear regression : $f: \mathbb{R}^D \rightarrow \mathbb{R}^C$

In this case, we have a multioutput model from a multidimensional input. Note that for a single point $\xvec \in \mathbb{R}^3$ and two outputs we have:

$$
\begin{split}
y_1 &= w_{11} x_1 + w_{21} x_2 + w_{31} x_3 + b_1\\
y_2 &= w_{12} x_1 + w_{22} x_2 + w_{32} x_3 + b_2\\
\end{split}
$$

This can be compactly expressed by noting that, when considering $N$ points and $C$ outputs, all the elements involved are now matrices.

$$
\begin{split}
\Ymat &= \Xmat\Wmat
\end{split}
$$


where:

$$
\begin{split}
\Xmat =
\begin{pmatrix}
x_1^{1} & x_2^{1} & x_3^{1} & 1 \\
x_1^{2} & x_2^{2} & x_3^{2} & 1 \\
\vdots ,& \vdots \\
x_1^{N} & x_2^{N} & x_3^{N} & 1 \\
\end{pmatrix}
\end{split}
$$


$$
\begin{split}
\Wmat =
\begin{pmatrix}
w_{11}, & w_{12}, & \dots & w_{1C}\\
w_{21}, & w_{22}, & \dots & w_{2C}\\
w_{31}, & w_{32}, & \dots & w_{3C}\\
b_{1} , & b_{2}, & \dots & b_{C}
\end{pmatrix}
\end{split}
$$


$$
\begin{split}
\Ymat =
\begin{pmatrix}
y_1^{1},& y_2^{1},  & \dots  & y_C^{1}\\
y_1^{2},&  y_2^{2}, & \dots & y_C^{2}\\
\vdots ,& \vdots,   & \vdots & \vdots \\
y_1^{N},& y_2^{N},  & \dots & y_C^{N} \\
\end{pmatrix}
\end{split}
$$

So since this can be exactly written down in the same way as the previous section, all the formulations and explanations are exactly the same. This is the magic of multivariate calculus and function compositions. Note that now, if we expand the loss function, we have:

$$
\begin{split}
L(\Wmat,\Xmat,\Tmat) &= \sum^N_{n=1}\sum^C_{c=1}(t_c^n - y_c^n)^2\\
  &= \sum^N_{n=1}\sum^C_{c=1}\left(t_c^n - \sum^D_{d=1} w_{dc}x_d^n - b_c\right)^2
\end{split}
$$

We start to see how the mathematical expressions involving loss functions start to look a bit ugly, and we will see they get even uglier when classification likelihoods come into place.

## Deep dive into function composition

(This section has been entirely created by Claude based on specific prompts from Juan that make references to all the previous derivations in this notebook, which are entirely done by Juan. There are some minor edits made by Juan that correct some math errors from Claude. The special note section is entirely done by Juan to showcase some differences. There was also one of the subsections that uses row vectors, and Claude used matrices. I changed it to row vectors.

Throughout this chapter, we have solved the same kind of problem again and again: obtaining a Jacobian (or gradient) by writing a loss as a composition of simple functions, and multiplying the individual Jacobians via the chain rule. In this section we want to make explicit something that has remained implicit so far: even when the *overall* function we care about is, say, $f:\mathbb{R}\rightarrow\mathbb{R}$, the composition that builds it up can (and typically does) travel through intermediate spaces that are **not** $\mathbb{R}\rightarrow\mathbb{R}$ at all — vectors, matrices, whatever shape the problem needs.

To keep things focused, we will always use the same generic four-step composition, and we will always work with the **squared loss** (the reasoning for the absolute loss is analogous):

$$
\begin{split}
\Ymat &= \Xmat\Wmat\\
\Zmat &= \Tmat-\Ymat \\
\Cmat &= \Zmat^2 \quad \text{element-wise}\\
L &= \onevect\Cmat\onevec
\end{split}
$$

The last step is nothing more than summing all the entries of $\Cmat$, but written as a double dot product with vectors of ones instead of with explicit summation signs — the same trick we used before. Note that the last two steps could be replaced by the trace operator, as we did previously. 

Also, to genuinely have cases where the parameter lives in $\mathbb{R}$ (and not $\mathbb{R}^2$), **we drop the bias term in every model below**: this way, for $f:\mathbb{R}\rightarrow\mathbb{R}$ the single weight $w$ is really the *only* parameter, so differentiating wrt it is truly a $\mathbb{R}\rightarrow\mathbb{R}$ problem end to end. Below, $\Xmat,\Wmat,\Tmat,\Ymat,\Zmat,\Cmat$ (or their vector/scalar counterparts) will take different concrete shapes in each case, but the underlying pattern is always the same one.

### $f:\mathbb{R}\rightarrow\mathbb{R}$, $N=1$

Here we have a single scalar weight $w$ (no bias) mapping a single scalar input $x$ to a single scalar output, compared against a single scalar target $t$. Both $x$ and $t$ are fixed data (constants); the only variable is $w$.

$$
\begin{align*}
y &= xw && \mathbb{R} \to \mathbb{R}\\
z &= t - y && \mathbb{R} \to \mathbb{R}\\
c &= z^2 && \mathbb{R} \to \mathbb{R}\\
L &= c && \mathbb{R} \to \mathbb{R}
\end{align*}
$$

Since $N=1$, there is nothing to sum: $L=c$ is just the identity, included only to keep the same four steps as every other case.

The Jacobians of each individual transformation are:

$$
\begin{align*}
\Jac{x}{y} &= x && \in \mathbb{R}^{1\times 1}\\
\Jac{y}{z} &= -1 && \in \mathbb{R}^{1\times 1}\\
\Jac{z}{c} &= 2z && \in \mathbb{R}^{1\times 1}\\
\Jac{c}{L} &= 1 && \in \mathbb{R}^{1\times 1}
\end{align*}
$$

And the full Jacobian, obtained by the chain rule as the product of all of the above, is:

$$
\Jac{w}{} L =\Jac{c}{L}\Jac{z}{c} \Jac{y}{z} \Jac{x}{y} = (1)(2z)(-1)(x) = -2x(t-xw) \quad \in \mathbb{R}^{1\times 1}
$$

### $f:\mathbb{R}\rightarrow\mathbb{R}$, $N=N$

Now we keep a single scalar weight $w$ (still no bias), but we have $N$ data points, stacked in $\xvec,\tvec\in\mathbb{R}^N$ (fixed constants). Note how the *domain* of the composition is still $\mathbb{R}$ (that of $w$), but the intermediate steps live in $\mathbb{R}^N$.

$$
\begin{align*}
\yvec &= \xvec w && \mathbb{R} \to \mathbb{R}^N\\
\zvec &= \tvec-\yvec && \mathbb{R}^N \to \mathbb{R}^N \\
\cvec &= \zvec^2 && \mathbb{R}^N \to \mathbb{R}^N \quad \text{element-wise}\\
L &= \onevect\cvec && \mathbb{R}^N \to \mathbb{R}
\end{align*}
$$

$$
\begin{align*}
\Jac{w}{\yvec} &= \xvec && \in \mathbb{R}^{N\times 1}\\
\Jac{\yvec}{\zvec} &= -\Imat && \in \mathbb{R}^{N\times N}\\
\Jac{\zvec}{\cvec} &= 2\diag(\zvec) && \in \mathbb{R}^{N\times N}\\
\Jac{\cvec}{L} &= \onevect && \in \mathbb{R}^{1\times N}
\end{align*}
$$

$$
\Jac{w}{} L =\Jac{\cvec}{L}\Jac{\zvec}{\cvec} \Jac{\yvec}{\zvec} \Jac{w}{\yvec} = \onevect \cdot 2\diag(\zvec) \cdot (-\Imat)\cdot \xvec = -2\onevect\diag(\tvec-\xvec w)\xvec \quad \in \mathbb{R}^{1\times 1}
$$

As you can check, this matches exactly the derivation we did earlier for the squared loss (just without the bias term).

### $f:\mathbb{R}^D\rightarrow\mathbb{R}$, $N=1$

Now the weight is a full vector $\wvec\in\mathbb{R}^D$ (no bias), the single data point is $\xvec\in\mathbb{R}^D$, and the target is a scalar $t$ (both fixed).

$$
\begin{align*}
y &= \xvect\wvec && \mathbb{R}^D \to \mathbb{R}\\
z &= t-y && \mathbb{R} \to \mathbb{R}\\
c &= z^2 && \mathbb{R} \to \mathbb{R}\\
L &= c && \mathbb{R} \to \mathbb{R}
\end{align*}
$$

$$
\begin{align*}
\Jac{\wvec}{y} &= \xvect && \in \mathbb{R}^{1\times D}\\
\Jac{y}{z} &= -1 && \in \mathbb{R}^{1\times 1}\\
\Jac{z}{c} &= 2z && \in \mathbb{R}^{1\times 1}\\
\Jac{c}{L} &= 1 && \in \mathbb{R}^{1\times 1}
\end{align*}
$$

$$
\Jac{\wvec}{} L = \Jac{c}{L}\Jac{z}{c}\Jac{y}{z}\Jac{\wvec}{y} = (1)(2z)(-1)(\xvect) = -2(t-\xvect\wvec)\xvect \quad \in \mathbb{R}^{1\times D}
$$

### $f:\mathbb{R}^D\rightarrow\mathbb{R}$, $N=N$

$\wvec\in\mathbb{R}^D$, data $\Xmat\in\mathbb{R}^{N\times D}$, $\tvec\in\mathbb{R}^N$ (fixed).

$$
\begin{align*}
\yvec &= \Xmat\wvec && \mathbb{R}^D \to \mathbb{R}^N\\
\zvec &= \tvec-\yvec && \mathbb{R}^N \to \mathbb{R}^N \\
\cvec &= \zvec^2 && \mathbb{R}^N \to \mathbb{R}^N \quad \text{element-wise}\\
L &= \onevect\cvec && \mathbb{R}^N \to \mathbb{R}
\end{align*}
$$

$$
\begin{align*}
\Jac{\wvec}{\yvec} &= \Xmat && \in \mathbb{R}^{N\times D}\\
\Jac{\yvec}{\zvec} &= -\Imat && \in \mathbb{R}^{N\times N}\\
\Jac{\zvec}{\cvec} &= 2\diag(\zvec) && \in \mathbb{R}^{N\times N}\\
\Jac{\cvec}{L} &= \onevect && \in \mathbb{R}^{1\times N}
\end{align*}
$$

$$
\Jac{\wvec}{} L = \Jac{\cvec}{L}\Jac{\zvec}{\cvec}\Jac{\yvec}{\zvec}\Jac{\wvec}{\yvec} = \onevect \cdot 2\diag(\zvec) \cdot(-\Imat)\cdot \Xmat = -2\onevect\diag(\tvec-\Xmat\wvec)\Xmat \quad \in \mathbb{R}^{1\times D}
$$

This is precisely the multivariate case we derived earlier (again, minus the bias).

### $f:\mathbb{R}\rightarrow\mathbb{R}^C$, $N=1$

Now, a single scalar input $x$ must produce $C$ outputs. Without a bias, this means one weight per output, $\wvec\in\mathbb{R}^C$, with $y_c = x\cdot w_c$. The single target is now a vector $\tvec\in\mathbb{R}^C$ (fixed).

$$
\begin{align*}
\yvec &= x\wvec && \mathbb{R}^C \to \mathbb{R}^C\\
\zvec &= \tvec-\yvec && \mathbb{R}^C \to \mathbb{R}^C \\
\cvec &= \zvec^2 && \mathbb{R}^C \to \mathbb{R}^C \quad \text{element-wise}\\
L &= \onevect\cvec && \mathbb{R}^C \to \mathbb{R}
\end{align*}
$$

$$
\begin{align*}
\Jac{\wvec}{\yvec} &= x\Imat && \in \mathbb{R}^{C\times C}\\
\Jac{\yvec}{\zvec} &= -\Imat && \in \mathbb{R}^{C\times C}\\
\Jac{\zvec}{\cvec} &= 2\diag(\zvec) && \in \mathbb{R}^{C\times C}\\
\Jac{\cvec}{L} &= \onevect && \in \mathbb{R}^{1\times C}
\end{align*}
$$

$$
\Jac{\wvec}{} L = \Jac{\cvec}{L}\Jac{\zvec}{\cvec}\Jac{\yvec}{\zvec}\Jac{\wvec}{\yvec} = \onevect \cdot 2\diag(\zvec) \cdot (-\Imat)\cdot x\Imat = -2x\,\onevect\diag(\tvec-x\wvec) \quad \in \mathbb{R}^{1\times C}
$$

### $f:\mathbb{R}\rightarrow\mathbb{R}^C$, $N=N$

$\wvec\in\mathbb{R}^C$ (one scalar weight per output), data $\xvec\in\mathbb{R}^N$ ($N$ scalar inputs) and $\Tmat\in\mathbb{R}^{N\times C}$ (fixed). Here $\Ymat$ is built from an outer product between $\xvec$ and $\wvec$.

$$
\begin{align*}
\Ymat &= \xvec\wvect && \mathbb{R}^C \to \mathbb{R}^{N\times C}\\
\Zmat &= \Tmat-\Ymat && \mathbb{R}^{N\times C} \to \mathbb{R}^{N\times C} \\
\Cmat &= \Zmat^2 && \mathbb{R}^{N\times C} \to \mathbb{R}^{N\times C} \quad \text{element-wise}\\
L &= \onevect\Cmat\onevec && \mathbb{R}^{N\times C} \to \mathbb{R}
\end{align*}
$$

As before, since $\Ymat$ (and $\Zmat,\Cmat$) are matrices but the parameter $\wvec$ is a vector, we work through the vectorized ($\vvec$) form of the differentials, exactly as we did for the multioutput squared loss earlier.

$$
\begin{align*}
\Jac{\wvec}{\vvec\Ymat} &= \Imat\otimes\xvec && \in \mathbb{R}^{NC\times C}\\
\Jac{\vvec\Ymat}{\vvec\Zmat} &= -\Imat && \in \mathbb{R}^{NC\times NC}\\
\Jac{\vvec\Zmat}{\vvec\Cmat} &= 2\diag(\vvec\Zmat) && \in \mathbb{R}^{NC\times NC}\\
\Jac{\vvec\Cmat}{L} &= \vvec\braT{\onevec\onevect} && \in \mathbb{R}^{1\times NC}
\end{align*}
$$

$$
\Jac{\wvec}{} L = \Jac{\vvec\Cmat}{L}\Jac{\vvec\Zmat}{\vvec\Cmat}\Jac{\vvec\Ymat}{\vvec\Zmat}\Jac{\wvec}{\vvec\Ymat} = \vvec\braT{\onevec\onevect}\cdot 2\diag(\vvec\Zmat)\cdot(-\Imat)\cdot(\Imat\otimes\xvec) = -2\,\vvec\braT{\onevec\onevect}\diag(\vvec\bra{\Tmat - \xvec\wvect})(\Imat\otimes\xvec) \quad \in \mathbb{R}^{1\times C}
$$

### $f:\mathbb{R}^D\rightarrow\mathbb{R}^C$, $N=1$

Now $\Wmat\in\mathbb{R}^{D\times C}$ (no bias), single data point $\Xmat=\xvect\in\mathbb{R}^{1\times D}$ and target $\tvect\in\mathbb{R}^{1\times C}$ (fixed). Because $N=1$, $\onevect$ is trivial (a $1\times 1$ identity), so it disappears from $L$ and we only keep $\onevec$ to sum over the $C$ outputs.

$$
\begin{align*}
\yvect &= \xvect\Wmat && \mathbb{R}^{D\times C} \to \mathbb{R}^{1\times C}\\
\zvect &= \tvect-\yvect && \mathbb{R}^{1\times C} \to \mathbb{R}^{1\times C} \\
\cvect &= \pare{\zvect}^2 && \mathbb{R}^{1\times C} \to \mathbb{R}^{1\times C} \quad \text{element-wise}\\
L &= \cvect\onevec && \mathbb{R}^{1\times C} \to \mathbb{R}
\end{align*}
$$

The Jacobians are:

$$
\begin{align*}
\Jac{\vvec\Wmat}{\yvec} &= \Imat\otimes\xvect && \in \mathbb{R}^{C\times DC}\\
\Jac{\yvec}{\zvec} &= -\Imat && \in \mathbb{R}^{C\times C}\\
\Jac{\zvec}{\cvec} &= 2\diag(\zvec) && \in \mathbb{R}^{C\times C}\\
\Jac{\cvec}{L} &= \onevect && \in \mathbb{R}^{1\times C}
\end{align*}
$$

$$
\Jac{\vvec\Wmat}{} L = \Jac{\cvec}{L}\Jac{\zvec}{\cvec}\Jac{\yvec}{\zvec}\Jac{\vvec\Wmat}{\yvec} = \onevect\cdot 2\diag(\zvec)\cdot(-\Imat)\cdot(\Imat\otimes\xvect) = -2\,\onevect\diag(\tvec - \Wmatt\xvec)(\Imat\otimes\xvect) \quad \in \mathbb{R}^{1\times DC}
$$


#### Special note here.

Here we have an example of how we can derive the Jacobian through different paths. In particular, for the function $\yvect = \xvect\Wmat$, we have worked out the canonical form by first taking vec on both sides. Since $\vvec{\yvect}=\vvec{\yvec}=\yvec$, this gives:

$$
\begin{split}
\dd \vvec{\yvect} &= \dd\bra{\vvec\bra{\xvect\Wmat}}\\
\dd \yvec &= \vvec{\xvect\dd \Wmat}\\
\dd \yvec &= \bra{\Imat \otimes \xvect}\dd\vvec\Wmat
\end{split}
$$ 

Another option is the one that is used in the rest of the Jacobians from this step. The canonical form of a vector-valued vector argument function is given within column vectors. Thus, we can start transposing on both sides, since the differential of transposed is the transpose of the differential. Afterward, take vec on both sides towards canonical form and use the commutation matrix (https://tminka.github.io/papers/matrix/minka-matrix.pdf), which says $K_{CD}\vvec\Wmat=\vvec\Wmatt$:

$$
\begin{split}
\dd \yvect &= \dd\bra{\xvect\Wmat}\\
\dd \yvec &= \dd\Wmatt\xvec\\
\dd \vvec{\yvec} &= \vvec{\dd\Wmatt\xvec}\\
\dd \yvec &= \bra{\xvect\otimes\Imat}\dd\vvec\Wmatt\\
\dd \yvec &= \bra{\xvect\otimes\Imat}K_{CD}\dd\vvec\Wmat
\end{split}
$$

So both Jacobians are equivalent. One thing is computing the Jacobian; the other is looking for the most efficient way to compute it. We will see an example of this in chapter 6. As a final thing to look at, we can change function composition here to:

$$
\begin{align*}
\yvec &= \Wmatt\xvec  \\
\zvec &= \tvec-\yvec \\
\cvec &= \pare{\zvec}^2  \quad \text{element-wise}\\
L &= \onevect\cvec 
\end{align*}
$$

This allows us to express everything in column vectors, so that canonical forms read out quicker. Note that directly targeting Jacobians here would give rise to our second derivation, the one using the commutation matrix. So here we have an example where changing the convention of $\Xmat$ having data as row vectors leads to a less efficient computation. We will apply this same change in chapter 6.


### $f:\mathbb{R}^D\rightarrow\mathbb{R}^C$, $N=N$ (general case)

Finally the fully general case: $\Wmat\in\mathbb{R}^{D\times C}$, $\Xmat\in\mathbb{R}^{N\times D}$, $\Tmat\in\mathbb{R}^{N\times C}$ (all fixed except $\Wmat$). This is exactly the composition we already used for the multi-output squared loss, just without the bias row.

$$
\begin{align*}
\Ymat &= \Xmat\Wmat && \mathbb{R}^{D\times C} \to \mathbb{R}^{N\times C}\\
\Zmat &= \Tmat-\Ymat && \mathbb{R}^{N\times C} \to \mathbb{R}^{N\times C} \\
\Cmat &= \Zmat^2 && \mathbb{R}^{N\times C} \to \mathbb{R}^{N\times C} \quad \text{element-wise}\\
L &= \onevect\Cmat\onevec && \mathbb{R}^{N\times C} \to \mathbb{R}
\end{align*}
$$

$$
\begin{align*}
\Jac{\vvec\Wmat}{\vvec\Ymat} &= \Imat\otimes\Xmat && \in \mathbb{R}^{NC\times DC}\\
\Jac{\vvec\Ymat}{\vvec\Zmat} &= -\Imat && \in \mathbb{R}^{NC\times NC}\\
\Jac{\vvec\Zmat}{\vvec\Cmat} &= 2\diag(\vvec\Zmat) && \in \mathbb{R}^{NC\times NC}\\
\Jac{\vvec\Cmat}{L} &= \vvec\braT{\onevec\onevect} && \in \mathbb{R}^{1\times NC}
\end{align*}
$$

$$
\Jac{\vvec\Wmat}{} L = \Jac{\vvec\Cmat}{L}\Jac{\vvec\Zmat}{\vvec\Cmat}\Jac{\vvec\Ymat}{\vvec\Zmat}\Jac{\vvec\Wmat}{\vvec\Ymat} = \vvec\braT{\onevec\onevect}\cdot 2\diag(\vvec\Zmat)\cdot(-\Imat)\cdot(\Imat\otimes\Xmat) = -2\,\vvec\braT{\onevec\onevect}\diag(\vvec\bra{\Tmat - \Xmat\Wmat})(\Imat\otimes\Xmat) \quad \in \mathbb{R}^{1\times DC}
$$

Notice that every one of the seven previous cases is nothing more than this exact same formula, specialized to $D=1$, $C=1$ and/or $N=1$: whenever one of these dimensions collapses to $1$, the corresponding identity matrix in the Kronecker product $\Imat\otimes\Xmat$ (or the vectors $\onevect,\onevec$) becomes trivial, and the expression reduces to the simpler forms we saw above. This is the concrete sense in which "composition of functions" is the same machinery regardless of whether we call a particular map $\mathbb{R}\to\mathbb{R}$, $\mathbb{R}^D\to\mathbb{R}$, or anything else — the shapes change, the reasoning doesn't.

## Scaling the loss function by a constant.

We previously mentioned that the loss functions being minimized are called the expected loss, since they come from an expectation that is hidden under the hood. In reality, scaling a function by some constant does not change the location of the minimum. This can be easily seen as follows. Consider any loss function $L$. The minimum is where the gradient takes the value of 0. This means that we solve:

$$
\begin{split}
\grad{x}{} f(x) = 0
\end{split}
$$

Note that since a scaling operation is linear, it can be interchanged with the gradient; this means that for any constant $C$:

$$
\begin{split}
\grad{x}{} \frac{1}{C} f(x) &= 0\\
\frac{1}{C} \grad{x}{}  f(x)&=0\\
\grad{x}{} f(x)&= C\cdot 0
\end{split}
$$


This result can be generalized to any monotonic transformation. If we have a transformation $g()$ that is monotonically increasing, then the minimum does not change. This includes functions such as the \log. So whenever we are dealing with machine learning problems that are based on optimizing a loss function, there is no difference in whether we scale the loss. I have just decided to present all this using $C=1$ for ease of exposition.

## TODO

* parameter sharing
* multicolineality
* non-independent outputs in the loss function (probably in the probabilistic machine learning section)
